# Alternate RNA Decoding Analysis Pipeline

## Generate All Dependencies for All 3 Datasets (Ping_2018, Takasugi_2024, Bai_2020)
## Run SAAP Detection, Quantification, Validation, and Analysis for all 3 Datasets

### Last Updated 05-18-2026 by Alex Maropakis
### Some code adapted from Tsour et al. Nature 2026 

# Generate Analysis Dependencies from 3 Datasets

In [ ]:
import os
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import re
from collections import Counter
import scipy as sp
from copy import deepcopy
from statsmodels.stats.multitest import multipletests
from adjustText import adjust_text
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator
import os
import ast
from scipy.odr import Model, RealData, ODR
from scipy.stats import gaussian_kde
from matplotlib import gridspec
#mport milkviz as mv
from matplotlib_venn import venn3
import zipfile
from glob import glob
import gzip
import tarfile
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoMinorLocator, FixedLocator
import statsmodels.api as sm
from statsmodels.formula.api import ols
import random
import copy
from typing import Any

### Setting directories and reading in data 

In [ ]:
print("Loading directories...")

CODE_DIR        = '/Users/alexmaropakis/Projects/BrainDecode/'
PROJECT_DIR     = '/Users/alexmaropakis/Projects/Project_BrainDecode/'
INDIR           = PROJECT_DIR + 'Analysis_Inputs/'
OUTDIR          = CODE_DIR + 'Dependencies/Analysis_Outputs/'
DEPDIR          = CODE_DIR + 'Dependencies/'
MQ_DIR          = PROJECT_DIR + 'mq_output/'
PLOT_DIR        = PROJECT_DIR + 'Plots/'
SAMPLE_MAPS_DIR = DEPDIR + 'Sample_maps/'

os.makedirs(DEPDIR, exist_ok=True)
os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

# ── Ping 2018 ────────────────────────────────────────────────────────────────
ping2018        = INDIR + 'Ping_2018/'
acg             = ping2018 + 'acg/'
acg_mq          = MQ_DIR + 'Ping_2018/acg'
acg_batches     = ['b1','b2','b3','b4','b5']
acg_dirs, acg_samples, acg_maps, acg_mq_dirs = [], [], [], []
for b in acg_batches:
    _map = pd.read_excel(SAMPLE_MAPS_DIR + f'sample_map_acg{b}.xlsx')
    acg_dirs.append(acg + f'{b}/')
    acg_maps.append(_map)
    acg_samples.append(['S' + str(i) for i in sorted(set[Any](_map['TMT plex']))])
    acg_mq_dirs.append(acg_mq + f'{b}/combined/txt/')

fc             = ping2018 + 'fc/'
fc_mq          = MQ_DIR + 'Ping_2018/fc'
fc_batches     = ['b1','b2','b3','b4','b5']
fc_dirs, fc_samples, fc_maps, fc_mq_dirs = [], [], [], []
for b in fc_batches:
    _map = pd.read_excel(SAMPLE_MAPS_DIR + f'sample_map_fc{b}.xlsx')
    fc_dirs.append(fc + f'{b}/')
    fc_maps.append(_map)
    fc_samples.append(['S' + str(i) for i in sorted(set[Any](_map['TMT plex']))])
    fc_mq_dirs.append(fc_mq + f'{b}/combined/txt/')

# ── Takasugi 2024 ────────────────────────────────────────────────────────────
tak2024         = INDIR + 'Takasugi_2024/'
ms              = tak2024 + 'aged_mouse/'
ms_mq           = MQ_DIR + 'Takasugi_2024/'
ms_tissues      = ['Aorta', 'Brain', 'Heart', 'Kidney', 'Liver', 'Muscle', 'Skin']
ms_dirs, ms_samples, ms_maps, ms_mq_dirs = [], [], [], []
for tissue in ms_tissues:
    _aas_dir = ms + f'{tissue}/'
    _map     = pd.read_excel(SAMPLE_MAPS_DIR + f'sample_map_{tissue.lower()}.xlsx')
    ms_dirs.append(_aas_dir)
    ms_maps.append(_map)
    ms_samples.append(['S' + str(i) for i in sorted(set[Any](_map['TMT plex']))])
    ms_mq_dirs.append(ms_mq + f'{tissue}/')

# ── Bai 2020 ─────────────────────────────────────────────────────────────────
pooled          = INDIR + 'Bai_2020/'
pooled_mq       = MQ_DIR + 'Bai_2020/'
pooled_map      = pd.read_excel(SAMPLE_MAPS_DIR + 'sample_map_pooled.xlsx')
pooled_samples  = ['S' + str(i) for i in sorted(set[Any](pooled_map['TMT plex']))]

# ── Master lists (parallel — one entry per dataset) ──────────────────────────
datasets = (
    [f'acg{b}' for b in acg_batches],
    [f'fc{b}'  for b in fc_batches],
    ['pooled'],
    ms_tissues,
)
all_datasets    = [d for grp in datasets for d in grp]

proj_dir_list   = acg_dirs    + fc_dirs    + [pooled]    + ms_dirs
mq_dir_list     = acg_mq_dirs + fc_mq_dirs + [pooled_mq] + ms_mq_dirs
samples_list    = acg_samples + fc_samples + [pooled_samples] + ms_samples
sample_map_list = acg_maps    + fc_maps    + [pooled_map]     + ms_maps

# ── TMT-set → dataset name mapping ───────────────────────────────────────────
# Key is matched as a substring of proj_dir.lower().
tmt_to_dataset = {
    '/acg/b1/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/acg/b2/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/acg/b3/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/acg/b4/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/acg/b5/': {'S1': 'acgb1', 'S2': 'acgb2', 'S3': 'acgb3', 'S4': 'acgb4', 'S5': 'acgb5'},
    '/fc/b2/':  {'S1': 'fcb1',  'S2': 'fcb2',  'S3': 'fcb3',  'S4': 'fcb4',  'S5': 'fcb5'},
    '/fc/b3/':  {'S1': 'fcb1',  'S2': 'fcb2',  'S3': 'fcb3',  'S4': 'fcb4',  'S5': 'fcb5'},
    '/fc/b4/':  {'S1': 'fcb1',  'S2': 'fcb2',  'S3': 'fcb3',  'S4': 'fcb4',  'S5': 'fcb5'},
    '/fc/b5/':  {'S1': 'fcb1',  'S2': 'fcb2',  'S3': 'fcb3',  'S4': 'fcb4',  'S5': 'fcb5'},
    '/bai_2020/': {'S1': 'pooled'},
    '/takasugi_2024/aorta/':  {'S1': 'Aorta'},
    '/takasugi_2024/brain/':  {'S2': 'Brain'},
    '/takasugi_2024/heart/':  {'S3': 'Heart'},
    '/takasugi_2024/kidney/': {'S4': 'Kidney'},
    '/takasugi_2024/liver/':  {'S5': 'Liver'},
    '/takasugi_2024/muscle/': {'S7': 'Muscle'},
    '/takasugi_2024/skin/':   {'S8': 'Skin'},
}

# ── Sample-type classifier rules (order matters: ADPD before AD) ─────────────
SAMPLE_TYPES = [
    # Ping 2018 — disease groups
    ('ADPD', 'ADPD'),
    ('AD',   'AD'),
    ('PD',   'PD'),
    ('CTRL', 'CTRL'),
    # Bai 2020 — disease groups
    ('HPC',  'HPC'),
    ('LPC',  'LPC'),
    ('MCI',  'MCI'),
    ('PSP',  'PSP'),
    # Takasugi 2024 — age timepoints
    ('t06mo', 't06mo'),
    ('t15mo', 't15mo'),
    ('t24mo', 't24mo'),
    ('t30mo', 't30mo'),
]

print("Directories loaded successfully!")
print(f"  Datasets registered: {datasets}")

## Generate analysis dependencies

Generates all required dependency files per experiment from MaxQuant output and
pipeline-derived dictionaries.

Outputs (saved to `DEPDIR/{experiment}/`):

1. `dataset_metrics.xlsx`
2. `Position_probability_fragment_ion_data.csv`
3. `Fragment_ion_dict.p`
4. `SAAP_precursor_reporter_quant.xlsx`
5. `MTP_sequences.fasta`
6. `Modified_peptide_filter_dict_DP2valSAAP.p`
7. `PTM_heatmap_dict.p`
8. `PTM_heatmap_data.xlsx`
9. `fragments_per_saap_4barplot_allDS.xlsx`

Last Updated 05-11-2026 by Alex Maropakis

In [ ]:
# Helper Functions

# ── Pickle compatibility shim for legacy numpy/pandas pickles ───────────────
class CompatUnpickler(pickle.Unpickler):
    REMAP = {
        'numpy.core._multiarray_umath': 'numpy._core._multiarray_umath',
        'numpy.core.multiarray':        'numpy._core.multiarray',
        'numpy.core.numeric':           'numpy._core.numeric',
        'numpy.core':                   'numpy._core',
        'pandas.core.indexes.numeric':  'pandas.core.indexes.base',
        'pandas.core.indexes.int64':    'pandas.core.indexes.base',
        'pandas.core.indexes.float64':  'pandas.core.indexes.base',
    }
    def find_class(self, module, name):
        return super().find_class(self.REMAP.get(module, module), name)

def compat_load(path):
    with open(path, 'rb') as f:
        return CompatUnpickler(f).load()

# ── Experiment / sample-type labels ─────────────────────────────────────────
EXPERIMENT_PREFIXES   = {'acg': 'Ping_2018/acg', 'fc': 'Ping_2018/fc', 'pooled': 'Bai_2020'}
DATASET_EXP_OVERRIDES = {t: 'Takasugi_2024' for t in ms_tissues}

def get_experiment_label(dataset_name):
    if dataset_name in DATASET_EXP_OVERRIDES:
        return DATASET_EXP_OVERRIDES[dataset_name]
    for prefix, experiment in EXPERIMENT_PREFIXES.items():
        if dataset_name.lower().startswith(prefix):
            return experiment
    return 'other'

def classify_sample(sample_name):
    for pattern, label in SAMPLE_TYPES:
        if pattern in sample_name:
            return label
    return 'Unknown'

def exp_dir(experiment):
    path = os.path.join(DEPDIR, experiment)
    os.makedirs(path, exist_ok=True)
    return path

def resolve_dataset_key(proj):
    d = proj.lower()
    for key in tmt_to_dataset:
        if key.lower() in d:
            return tmt_to_dataset[key]
    return {}

In [ ]:
print("=== Generating: dataset_metrics.xlsx ===")

rows = []
for ds, mq_dir in zip(all_datasets, mq_dir_list):
    msms     = pd.read_csv(mq_dir + 'msms.txt',     sep='\t', low_memory=False)
    peptides = pd.read_csv(mq_dir + 'peptides.txt', sep='\t', low_memory=False)
    evidence = pd.read_csv(mq_dir + 'evidence.txt', sep='\t', low_memory=False)

    rows.append([
        ds,
        get_experiment_label(ds),
        msms[msms['PEP'] < 0.01].shape[0],
        peptides['Sequence'].nunique(),
        evidence['Sequence'].nunique(),
        peptides[peptides['Unique (Proteins)'] == 'yes']['Sequence'].nunique(),
    ])
    print(f"  {ds}: {rows[-1][2]:,} PSMs | {rows[-1][3]:,} peptides")

metrics_all = pd.DataFrame(rows, columns=[
    'Dataset', 'Experiment', 'Identified PSMs (1% FDR)',
    'Peptides', 'Peptides (evidence)', 'Unique peptides',
])

for exp, grp in metrics_all.groupby('Experiment'):
    out = os.path.join(exp_dir(exp), 'dataset_metrics.xlsx')
    grp.drop(columns='Experiment').reset_index(drop=True).to_excel(
        out, index=False, engine='openpyxl')
    print(f"  Saved → {out}")


In [ ]:
print("=== Generating: Position_probability_fragment_ion_data.csv + Fragment_ion_dict.p ===")

POS_COLS = [
    'SAAP', 'BP', 'AAS', 'AAS index', 'Positional probability', 'charge',
    'TMT set', 'Dataset',
    'saap_b_left_frag',  'saap_b_left_frag_mass',
    'saap_b_right_frag', 'saap_b_right_frag_mass',
    'saap_y_left_frag',  'saap_y_left_frag_mass',
    'saap_y_right_frag', 'saap_y_right_frag_mass',
    'bp_b_left_frag',    'bp_b_left_frag_mass',
    'bp_b_right_frag',   'bp_b_right_frag_mass',
    'bp_y_left_frag',    'bp_y_left_frag_mass',
    'bp_y_right_frag',   'bp_y_right_frag_mass',
]
FRAG_NAME_COLS = [c for c in POS_COLS if c.endswith('_frag')]
FRAG_MASS_COLS = [c for c in POS_COLS if c.endswith('_frag_mass')]

exp_rows, exp_frag, exp_count = {}, {}, {}

for ds, proj, samples, mq_dir in zip(all_datasets, proj_dir_list, samples_list, mq_dir_list):
    exp = get_experiment_label(ds)
    exp_rows.setdefault(exp, [])
    exp_frag.setdefault(exp, {})
    exp_count.setdefault(exp, 0)

    mtp_dict = compat_load(proj + 'Ion_validated_MTP_dict.p')
    msms     = pd.read_csv(mq_dir + 'msms.txt', sep='\t', low_memory=False)

    for s in samples:
        if s not in mtp_dict:
            continue
        s_df = pd.DataFrame.from_dict(mtp_dict[s])[
            ['mistranslated sequence', 'DP Base Sequence', 'aa subs',
             'origin aa index', 'aa subs positional probability', 'Charge']
        ].copy()
        s_df['TMT set'] = s
        s_df['Dataset'] = ds
        s_df.reset_index(drop=True, inplace=True)
        for c in FRAG_NAME_COLS: s_df[c] = ''
        for c in FRAG_MASS_COLS: s_df[c] = np.nan

        for i, row in s_df.iterrows():
            saap, bp, idx = row['mistranslated sequence'], row['DP Base Sequence'], row['origin aa index']
            global_i = i + exp_count[exp]
            exp_frag[exp][global_i] = {}

            for peptide, mapping, frag_key in [
                (saap, [
                    ('b' + str(idx),                  'saap_b_left'),
                    ('b' + str(idx + 1),              'saap_b_right'),
                    ('y' + str(len(saap) - idx + 1),  'saap_y_left'),
                    ('y' + str(len(saap) - idx),      'saap_y_right'),
                ], 'SAAP'),
                (bp, [
                    ('b' + str(idx),                  'bp_b_left'),
                    ('b' + str(idx + 1),              'bp_b_right'),
                    ('y' + str(len(bp) - idx + 1),    'bp_y_left'),
                    ('y' + str(len(bp) - idx),        'bp_y_right'),
                ], 'BP'),
            ]:
                hits = msms[msms['Sequence'] == peptide].sort_values(
                    'Number of matches', ascending=False)
                if hits.empty:
                    continue
                best   = hits.iloc[0]
                names  = best['Matches'].split(';')
                masses = best['Masses'].split(';')
                frags  = {n: m for n, m in zip(names, masses)
                          if ('b' in n or 'y' in n) and '-' not in n}
                exp_frag[exp][global_i][frag_key] = frags
                for frag_name, col_prefix in mapping:
                    s_df.loc[i, col_prefix + '_frag'] = frag_name
                    if frag_name in frags:
                        s_df.loc[i, col_prefix + '_frag_mass'] = float(frags[frag_name])

        exp_rows[exp].append(s_df)
        exp_count[exp] += len(s_df)

for exp, dfs in exp_rows.items():
    if not dfs:
        continue
    pos_df = pd.concat(dfs)
    pos_df.columns = POS_COLS

    edir     = exp_dir(exp)
    csv_path = os.path.join(edir, 'Position_probability_fragment_ion_data.csv')
    pkl_path = os.path.join(edir, 'Fragment_ion_dict.p')

    pos_df.to_csv(csv_path)
    with open(pkl_path, 'wb') as f:
        pickle.dump(exp_frag[exp], f)
    print(f"  Saved → {csv_path}")
    print(f"  Saved → {pkl_path}")


In [ ]:
print("=== Generating: SAAP_precursor_reporter_quant.xlsx ===")

pos_prob_all=pd.concat([
    pd.read_csv(os.path.join(DEPDIR,exp,'Position_probability_fragment_ion_data.csv'),index_col=0)
    for exp in {get_experiment_label(d) for d in all_datasets}
    if os.path.exists(os.path.join(DEPDIR,exp,'Position_probability_fragment_ion_data.csv'))
],ignore_index=True)

rows=[]
for ds,proj,mq_dir in zip(all_datasets,proj_dir_list,mq_dir_list):
    MTP_quant_dict=compat_load(proj+'MTP_quant_dict.p')
    ev=pd.read_csv(mq_dir+'evidence.txt',sep='\t',low_memory=False)
    ev['_pep']=ev['Sequence'].astype(str).str.strip().str.upper()
    pep_lookup=ev.groupby('_pep')['PEP'].min().to_dict()
    ds_map=resolve_dataset_key(proj)

    for entry in MTP_quant_dict.values():
        mtp_seq=str(entry['MTP_seq']).strip().upper()
        bp_seq=str(entry['BP_seq']).strip().upper()
        aa_sub=entry['aa_sub']
        tmt_sets=entry['tmt_sets']
        saap_pep=pep_lookup.get(mtp_seq,np.nan)
        bp_pep=pep_lookup.get(bp_seq,np.nan)

        for sample,sd in entry['Patient_dict'].items():
            for tmt_set in tmt_sets:
                dataset=ds_map.get(tmt_set,'Unknown')

                m=pos_prob_all[
                    (pos_prob_all['SAAP']==mtp_seq)&
                    (pos_prob_all['BP']==bp_seq)&
                    (pos_prob_all['TMT set']==tmt_set)&
                    (pos_prob_all['Dataset']==dataset)
                ]

                if m.empty:
                    continue

                rows.append({
                    'Dataset':dataset,
                    'Experiment':get_experiment_label(dataset),
                    'MTP_seq':mtp_seq,
                    'BP_seq':bp_seq,
                    'aa_sub':aa_sub,
                    'tmt_set':tmt_set,
                    'Positional_probability':m.iloc[0]['Positional probability'],
                    'SAAP_PEP':saap_pep,
                    'BP_PEP':bp_pep,
                    'MTP_PrecInt':entry['MTP_PrecInt'].get(tmt_set),
                    'BP_PrecInt':entry['BP_PrecInt'].get(tmt_set),
                    'Prec_RAAS':entry['Prec_ratio'].get(tmt_set),
                    'Sample':sample,
                    'Sample Type':classify_sample(sample),
                    'MTP_ReportInt':sd['MTP_ReportInt'],
                    'BP_ReportInt':sd['BP_ReportInt'],
                    'Reporter_RAAS':sd['Reporter_ratio'],
                    'BP_ReportInt_Norm':sd['BP_ReportInt_Norm'],
                    'MTP_ReportInt_Norm':sd['MTP_ReportInt_Norm'],
                })

saap_df=pd.DataFrame(rows)
saap_df.insert(0,'Row Number',range(len(saap_df)))

for exp,grp in saap_df.groupby('Experiment'):
    grp=grp.drop(columns='Experiment').reset_index(drop=True)
    grp['Row Number']=range(len(grp))
    out=os.path.join(exp_dir(exp),'SAAP_precursor_reporter_quant.xlsx')
    grp.to_excel(out,index=False)
    print(f"  Saved → {out}  ({len(grp):,} rows)")

In [ ]:
print("=== Generating: MTP_sequences.fasta ===")

exp_lines = {}
for ds, proj in zip(all_datasets, proj_dir_list):
    MTP_dict = compat_load(proj + 'MTP_quant_dict.p')
    exp = get_experiment_label(ds)
    exp_lines.setdefault(exp, [])
    for idx, entry in MTP_dict.items():
        mtp_seq = str(entry['MTP_seq']).strip().upper()
        aa_sub  = str(entry['aa_sub']).replace(',', '_').replace(' ', '')
        exp_lines[exp].extend([f">{ds}_MTP_{idx}_AAsub_{aa_sub}", mtp_seq])

for exp, lines in exp_lines.items():
    out = os.path.join(exp_dir(exp), 'MTP_sequences.fasta')
    with open(out, 'w') as f:
        f.write('\n'.join(lines))
    print(f"  Saved → {out}  ({len(lines)//2:,} sequences)")


In [ ]:
print("=== Generating: Modified_peptide_filter_dict_DP2valSAAP.p ===")

FILTER_FILES = [
    ('DP_search_evidence_dict.p', 'main', 'Raw file'),
    ('DP_dict.p',                 'dp',   'Raw file'),
    ('PTM_dict.p',                'ptm',  'Raw file'),
    ('MTP_dict.p',                'mtp',  'Raw file'),
    ('qMTP_dict.p',               'qmtp', 'mistranslated sequence'),
    ('Ion_validated_MTP_dict.p',  'ion',  'Raw file'),
]

exp_data = {}
for ds, proj, samples in zip(all_datasets, proj_dir_list, samples_list):
    parts = {key: compat_load(proj + fname) for fname, key, _ in FILTER_FILES}

    rows = [[s] + [len(parts[key][s][col]) for _, key, col in FILTER_FILES]
            for s in samples]
    df = pd.DataFrame(rows, columns=[
        'TMT set', 'Main peptides', 'DP', 'PTM', 'AAS',
        'High-confidence', 'Validated',
    ])
    df['Dataset'] = ds
    exp_data.setdefault(get_experiment_label(ds), {})[ds] = df

for exp, ds_dfs in exp_data.items():
    out = os.path.join(exp_dir(exp), 'Modified_peptide_filter_dict_DP2valSAAP.p')
    with open(out, 'wb') as f:
        pickle.dump(ds_dfs, f)
    print(f"  Saved → {out}")


In [ ]:
print("=== Generating: PTM_heatmap_dict.p + PTM_heatmap_data.xlsx ===")

TOP_N     = 40
N_SAMPLES = 23

def build_ptm_count_df(ptm_dict):
    master = []
    for v in ptm_dict.values():
        for ptms in v['PTM'].values():
            for p in ptms:
                if p not in master:
                    master.append(p)
    df = pd.DataFrame(index=master, columns=range(1, N_SAMPLES + 1))
    for s, v in ptm_dict.items():
        try:    s_int = int(s[1:])
        except: s_int = s
        cnt = Counter(p for sub in v['PTM'].values() for p in sub)
        for p, c in cnt.items():
            df.loc[p, s_int] = c
    return df.fillna(0).astype(float)

exp_ptm = {}
for ds, proj in zip(all_datasets, proj_dir_list):
    exp = get_experiment_label(ds)
    df  = build_ptm_count_df(compat_load(proj + 'PTM_dict.p'))
    df['avg'] = df.mean(axis=1)
    df.sort_values('avg', ascending=False, inplace=True)
    df.index = [x[0].upper() + x[1:] for x in df.index]
    exp_ptm.setdefault(exp, {})[ds] = df

for exp, ptm_heatmap_dict in exp_ptm.items():
    edir = exp_dir(exp)

    pkl_path = os.path.join(edir, 'PTM_heatmap_dict.p')
    with open(pkl_path, 'wb') as f:
        pickle.dump(ptm_heatmap_dict, f)
    print(f"  Saved → {pkl_path}")

    metrics = pd.read_excel(os.path.join(edir, 'dataset_metrics.xlsx'))

    top = None
    for df in ptm_heatmap_dict.values():
        s   = set(df.index[:TOP_N])
        top = s if top is None else top & s
    if not top:
        print(f"  WARNING: no shared top-{TOP_N} PTMs in '{exp}' — skipping data file")
        continue

    plot_df = pd.DataFrame({
        ds: ptm_heatmap_dict[ds].loc[list(top), 'avg']
        for ds in ptm_heatmap_dict
    }).astype(float)

    for ds in plot_df.columns:
        n = metrics.loc[metrics['Dataset'] == ds, 'Peptides (evidence)'].values
        if len(n):
            plot_df[ds] = plot_df[ds] / (n[0] / 1000)

    xlsx_path = os.path.join(edir, 'PTM_heatmap_data.xlsx')
    plot_df.to_excel(xlsx_path)
    print(f"  Saved → {xlsx_path}")


In [ ]:
print("=== Generating: by_fragments_per_saap_4barplot_allDS.xlsx ===")

CATS = ['0', '1', '2-5', '6-10', '>10']

def n_frags_over_mtp(matches, mtp, sub_idx):
    count = 0
    for f in matches:
        if any(t in f for t in ('NH3', 'H2O', '(', 'a')):
            continue
        if 'b' in f and int(f[1:]) > sub_idx:
            count += 1
        elif 'y' in f and len(mtp) - int(f[1:]) <= sub_idx:
            count += 1
    return count

exp_counts = {}
for ds, proj, samples, mq_dir in zip(all_datasets, proj_dir_list, samples_list, mq_dir_list):
    exp = get_experiment_label(ds)
    exp_counts.setdefault(exp, [0, 0, 0, 0, 0])

    val_path  = proj + 'Validated_MTP_dict.p'
    mtp_dict  = compat_load(val_path)
    ev_dict   = compat_load(proj + 'Validation_search_evidence_dict.p')
    msms      = pd.read_csv(mq_dir + 'msms.txt', sep='\t', low_memory=False)
    msms_vals = msms.values

    print(f"  Processing {ds}")

    for s in samples:
        if s not in mtp_dict:
            continue
        ev = ev_dict[s]
        mtp_dict[s]['fragment_evidence'] = {}

        for k in mtp_dict[s]['aa subs']:
            seq     = mtp_dict[s]['mistranslated sequence'][k]
            bp      = mtp_dict[s]['DP Base Sequence'][k]
            sub_idx = next(i for i, x in enumerate(bp) if seq[i] != x)
            ev_idx  = mtp_dict[s]['idx_val_evidence'][k]
            best    = 0

            for idx in ev_idx:
                row  = ev.iloc[idx]
                raw  = row['Raw file']
                scan = row['MS/MS scan number']
                hits = [i for i in range(len(msms_vals))
                        if msms_vals[i][0] == raw and msms_vals[i][1] == scan]
                if not hits:
                    continue
                matches_entry = msms.iloc[hits[0]]['Matches']
                if isinstance(matches_entry, str) and matches_entry.strip():
                    n = n_frags_over_mtp(matches_entry.split(';'), seq, sub_idx)
                    if n > best:
                        best = n
            mtp_dict[s]['fragment_evidence'][k] = best

    with open(val_path, 'wb') as f:
        pickle.dump(mtp_dict, f)

    for s in samples:
        if s not in mtp_dict:
            continue
        for n in mtp_dict[s].get('fragment_evidence', {}).values():
            if   n == 0:  exp_counts[exp][0] += 1
            elif n == 1:  exp_counts[exp][1] += 1
            elif n <= 5:  exp_counts[exp][2] += 1
            elif n <= 10: exp_counts[exp][3] += 1
            else:         exp_counts[exp][4] += 1

for exp, counts in exp_counts.items():
    out = os.path.join(exp_dir(exp), 'by_fragments_per_saap_4barplot_allDS.xlsx')
    df  = pd.DataFrame(zip(CATS, counts), columns=['Bin', 'Count'])
    df['Used'] = ['No' if b in ('0', '1') else 'Yes' for b in df['Bin']]
    df.to_excel(out)
    print(f"  Saved → {out}")


In [ ]:
print("=== Generating genome_substr_dict.p files ===")

GENOME_SUBSTR_KEYS = [
    '1-frame genome substring',
    '2-frame genome substring',
    '3-frame genome substring',
    '4-frame genome substring',
    '5-frame genome substring',
    '6-frame genome substring',
]

for proj_dir in proj_dir_list:
    dp_path = os.path.join(proj_dir, 'DP_dict.p')
    if not os.path.exists(dp_path):
        print(f"Skipping missing DP_dict.p: {dp_path}")
        continue

    print(f"Processing: {dp_path}")
    try:
        dp_dict = pickle.load(open(dp_path, 'rb'))
        genome_substr_dict = {}
        all_mtps = set()
        frame_hits = {
            1: set(),
            2: set(),
            3: set(),
            4: set(),
            5: set(),
            6: set(),
        }

        # dp_dict structure:
        # dp_dict[sample][field][index]
        for sample in dp_dict.keys():
            mt_seq_dict = dp_dict[sample]['mistranslated sequence']
            for idx, seq_list in mt_seq_dict.items():
                all_mtps.update(seq_list)
                for frame in range(1, 7):
                    key = f'{frame}-frame genome substring'
                    if key not in dp_dict[sample]:
                        continue
                    hit_list = dp_dict[sample][key].get(idx, [])
                    for seq_i, seq in enumerate(seq_list):
                        if (
                            seq_i < len(hit_list)
                            and hit_list[seq_i] == True
                        ):
                            frame_hits[frame].add(seq)

        genome_substr_dict = {
            'MTPs': list(all_mtps),

            'all_frame1_seqs': list(frame_hits[1]),
            'N_all_frame1': len(frame_hits[1]),

            'all_frame2_seqs': list(frame_hits[2]),
            'N_all_frame2': len(frame_hits[2]),

            'all_frame3_seqs': list(frame_hits[3]),
            'N_all_frame3': len(frame_hits[3]),

            'all_frame4_seqs': list(frame_hits[4]),
            'N_all_frame4': len(frame_hits[4]),

            'all_frame5_seqs': list(frame_hits[5]),
            'N_all_frame5': len(frame_hits[5]),

            'all_frame6_seqs': list(frame_hits[6]),
            'N_all_frame6': len(frame_hits[6]),

            # backwards compatibility with older decode notebooks
            'all_1frame_seqs': list(frame_hits[1]),
            'N_all_1frame': len(frame_hits[1]),

            'all_3frame_seqs': list(frame_hits[3]),
            'N_all_3frame': len(frame_hits[3]),

            'all_6frame_seqs': list(frame_hits[6]),
            'N_all_6frame': len(frame_hits[6]),
        }

        out_path = os.path.join(proj_dir, 'genome_substr_dict.p')
        pickle.dump(
            genome_substr_dict,
            open(out_path, 'wb')
        )
        print(f"Saved: {out_path}")

    except Exception as e:
        print(f"ERROR processing {proj_dir}")
        print(str(e))

print("Finished generating genome_substr_dict.p files.")

# Mouse Aging Analysis
### Dataset: Takasugi et al., Nature 2024

In [ ]:
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import re
from collections import Counter
import scipy as sp
from copy import deepcopy
from statsmodels.stats.multitest import multipletests
from adjustText import adjust_text
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator
import os
import ast
from scipy.odr import Model, RealData, ODR
from scipy.stats import gaussian_kde
from matplotlib import gridspec
from matplotlib_venn import venn3
import zipfile
from glob import glob
import gzip
import tarfile
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoMinorLocator, FixedLocator
import statsmodels.api as sm
from statsmodels.formula.api import ols
import matplotlib.patches as mpatches
from scipy.stats import ttest_ind
from itertools import combinations
from scipy.stats import fisher_exact, kruskal, mannwhitneyu

print("packages have been successfully read in!")

In [ ]:
# Functions for figure formatting

class GridShader():
    """
    function used to create alternating vertical gray and white background in plots
    """
    def __init__(self, ax, first=True, **kwargs):
        self.spans = []
        self.sf = first
        self.ax = ax
        self.kw = kwargs
        self.ax.autoscale(False, axis="x")
        self.cid = self.ax.callbacks.connect('xlim_changed', self.shade)
        self.shade()
    def clear(self):
        for span in self.spans:
            try:
                span.remove()
            except:
                pass
    def shade(self, evt=None):
        self.clear()
        xticks = self.ax.get_xticks()
        xlim = self.ax.get_xlim()
        xticks = xticks[(xticks > xlim[0]) & (xticks < xlim[-1])]
        locs = np.concatenate(([[xlim[0]], xticks, [xlim[-1]]]))

        start = [x-0.5 for x in locs[1-int(self.sf)::2]]
        end = [x-0.5 for x in locs[2-int(self.sf)::2]]

        for s, e in zip(start, end):
            self.spans.append(self.ax.axvspan(s, e, zorder=0, **self.kw))


def bihist(y1, y2, nbins=10, h=None):
    '''
    Function used to create violin plots as bihistograms with no smoothing.
    h is an axis handle. If not present, a new figure is created.
    '''
    if h is None: h = plt.figure().add_subplot(111)
    ymin = np.floor(np.minimum(min(y1), min(y2)))
    ymax = np.ceil(np.maximum(max(y1), max(y2)))
    bins = np.linspace(ymin, ymax, nbins)
    n1, bins1, patch1 = h.hist(y1, bins, orientation='horizontal', color='#aaaaaa', edgecolor=None, linewidth=0,rwidth=1)
    n2, bins2, patch2 = h.hist(y2, bins, orientation='horizontal', color='#aaaaaa', edgecolor=None, linewidth=0,rwidth=1)
    # set xmax:
    xmax = 0
    for i in patch1:
        i.set_edgecolor(None)
        width = i.get_width()
        if width > xmax: xmax = width
    # invert second histogram and set xmin:
    xmin = 0
    for i in patch2:
        i.set_edgecolor(None)
        width = i.get_width()
        width = -width
        i.set_width(width)
        if width < xmin: xmin = width
    h.set_xlim(xmin*1.1, xmax*1.1)          
    h.figure.canvas.draw()

In [ ]:
#Setting directories and reading in data 
#Reflect output from AAS pipeline (proj_dir), output folder (aa_subs_dir), and sample map for each sample 

print("loading directories...")
CODE_DIR        = '/Users/alexmaropakis/Projects/BrainDecode/'
PROJECT_DIR     = '/Users/alexmaropakis/Projects/Project_BrainDecode/'
INDIR           = PROJECT_DIR + 'Analysis_Inputs/Takasugi_2024/'
OUTDIR          = CODE_DIR + 'Dependencies/Analysis_Outputs/Takasugi_2024/'
DEPDIR          = CODE_DIR + 'Dependencies/Takasugi_2024/'
MQ_DIR          = PROJECT_DIR + 'mq_output/Takasugi_2024/'
PLOT_DIR        = PROJECT_DIR + 'Plots/Takasugi_2024/'
SAMPLE_MAPS_DIR = CODE_DIR + 'Dependencies/Sample_maps/'

os.makedirs(OUTDIR, exist_ok=True)
os.makedirs(PLOT_DIR, exist_ok=True)

#S1 AORTA
aorta_proj_dir      = OUTDIR + 'Aorta/'
aorta_aa_subs_dir   = INDIR + 'Aorta/'
aorta_sample_map    = pd.read_excel(SAMPLE_MAPS_DIR + 'sample_map_aorta.xlsx')
aorta_samples       = ['S'+str(i) for i in list(set(aorta_sample_map['TMT plex']))]
aorta_mq            = MQ_DIR + 'Aorta/'

#S2 BRAIN 
brain_proj_dir      = OUTDIR + 'Brain/'
brain_aa_subs_dir   = INDIR + 'Brain/'
brain_sample_map    = pd.read_excel(SAMPLE_MAPS_DIR + 'sample_map_brain.xlsx')
brain_samples       = ['S'+str(i) for i in list(set(brain_sample_map['TMT plex']))]
brain_mq            = MQ_DIR + 'Brain/'

#S3 HEART
heart_proj_dir      = OUTDIR + 'Heart/'
heart_aa_subs_dir   = INDIR + 'Heart/'
heart_sample_map    = pd.read_excel(SAMPLE_MAPS_DIR + 'sample_map_heart.xlsx')
heart_samples       = ['S'+str(i) for i in list(set(heart_sample_map['TMT plex']))]
heart_mq            = MQ_DIR + 'Heart/'

#S4 KIDNEY
kidney_proj_dir      = OUTDIR + 'Kidney/'
kidney_aa_subs_dir   = INDIR + 'Kidney/'
kidney_sample_map    = pd.read_excel(SAMPLE_MAPS_DIR + 'sample_map_kidney.xlsx')
kidney_samples       = ['S'+str(i) for i in list(set(kidney_sample_map['TMT plex']))]
kidney_mq            = MQ_DIR + 'Kidney/'

#S5 LIVER
liver_proj_dir      = OUTDIR + 'Liver/'
liver_aa_subs_dir   = INDIR + 'Liver/'
liver_sample_map    = pd.read_excel(SAMPLE_MAPS_DIR + 'sample_map_liver.xlsx')
liver_samples       = ['S'+str(i) for i in list(set(liver_sample_map['TMT plex']))]
liver_mq            = MQ_DIR + 'Liver/'

#S6 LUNG
# lung_proj_dir      = OUTDIR + 'Lung/'
# lung_aa_subs_dir   = INDIR + 'Lung/'
# lung_sample_map    = pd.read_excel(SAMPLE_MAPS_DIR + 'sample_map_lung.xlsx')
# lung_samples       = ['S'+str(i) for i in list(set(lung_sample_map['TMT plex']))]
# lung_mq            = MQ_DIR + 'Lung/'

#S7 MUSCLE
muscle_proj_dir      = OUTDIR + 'Muscle/'
muscle_aa_subs_dir   = INDIR + 'Muscle/'
muscle_sample_map    = pd.read_excel(SAMPLE_MAPS_DIR + 'sample_map_muscle.xlsx')
muscle_samples       = ['S'+str(i) for i in list(set(muscle_sample_map['TMT plex']))]
muscle_mq            = MQ_DIR + 'Muscle/'

#S8 SKIN
skin_proj_dir      = OUTDIR + 'Skin/'
skin_aa_subs_dir   = INDIR + 'Skin/'
skin_sample_map    = pd.read_excel(SAMPLE_MAPS_DIR + 'sample_map_skin.xlsx')
skin_samples       = ['S'+str(i) for i in list(set(skin_sample_map['TMT plex']))]
skin_mq            = MQ_DIR + 'Skin/'

#lists to quickly index dataset-specific sample maps from just the dataset name

## lists with lung tissue added 
# datasets = ['Aorta', 'Brain', 'Heart', 'Kidney', 'Liver', 'Lung', 'Muscle', 'Skin']
# data_dir_list = [aorta_aa_subs_dir, brain_aa_subs_dir, heart_aa_subs_dir, kidney_aa_subs_dir, liver_aa_subs_dir, lung_aa_subs_dir, muscle_aa_subs_dir, skin_aa_subs_dir]
# samples_list = [aorta_samples, brain_samples, heart_samples, kidney_samples, liver_samples, lung_samples, muscle_samples, skin_samples]
# sample_map_list = [aorta_sample_map, brain_sample_map, heart_sample_map, kidney_sample_map, liver_sample_map, lung_sample_map, muscle_sample_map, skin_sample_map]
# proj_dir_list = [aorta_proj_dir, brain_proj_dir, heart_proj_dir, kidney_proj_dir, liver_proj_dir, lung_proj_dir, muscle_proj_dir, skin_proj_dir]
# mq_dir_list = [aorta_mq, brain_mq, heart_mq, kidney_mq, liver_mq, lung_mq, muscle_mq, skin_mq]

datasets = ['Aorta', 'Brain', 'Heart', 'Kidney', 'Liver', 'Muscle', 'Skin']
data_dir_list = [aorta_aa_subs_dir, brain_aa_subs_dir, heart_aa_subs_dir, kidney_aa_subs_dir, liver_aa_subs_dir, muscle_aa_subs_dir, skin_aa_subs_dir]
samples_list = [aorta_samples, brain_samples, heart_samples, kidney_samples, liver_samples, muscle_samples, skin_samples]
sample_map_list = [aorta_sample_map, brain_sample_map, heart_sample_map, kidney_sample_map, liver_sample_map, muscle_sample_map, skin_sample_map]
proj_dir_list = [aorta_proj_dir, brain_proj_dir, heart_proj_dir, kidney_proj_dir, liver_proj_dir, muscle_proj_dir, skin_proj_dir]
mq_dir_list = [aorta_mq, brain_mq, heart_mq, kidney_mq, liver_mq, muscle_mq, skin_mq]


print("directories loaded successfully!")

In [ ]:
""" 
Read in data generated from above dependencies code
if dependencies have not been generated, go back and generate
""" 

pos_prob_df = pd.read_csv(DEPDIR+'Position_probability_fragment_ion_data.csv', index_col=0)
print(pos_prob_df)
frag_dict = pickle.load(open(DEPDIR+'Fragment_ion_dict.p', 'rb')) ### keys of frag_dict refer to the index in pos_prob_df
print(frag_dict)

SAAP_quant_df = pd.read_excel(DEPDIR+'SAAP_precursor_reporter_quant_data.xlsx', index_col=0)
#SAAP_precursor_reporter_quant_data.xlsx contains both precursor and reporter ion data for validated SAAPs
print(SAAP_quant_df)
SAAP_quant_df_list = [SAAP_quant_df.loc[SAAP_quant_df['Dataset']==ds] for ds in datasets]

In [ ]:
### Troubleshooting Panel
# Optional code to open pickle files and inspect formatting/population
# use this code to look at the formatting/contents of pipeline output files

"""
# data_dir = "/Users/alexa/OneDrive/Documents/2025_Maropakis_AAS/pipeline_output/Aorta/AA_subs_pipeline/"
# dp_ev_dict = pickle.load(open(data_dir+'DP_search_evidence_dict.p', 'rb'))
# dp_dict = pickle.load(open(data_dir+'DP_dict.p', 'rb'))
# mtp_dict = pickle.load(open(data_dir+'MTP_dict.p', 'rb'))
# ptm_dict = pickle.load(open(data_dir+'PTM_dict.p', 'rb'))
# hc_mtp_dict = pickle.load(open(data_dir+'MTP_quant_dict.p', 'rb'))
# qMTP = pickle.load(open(data_dir+'qMTP_dict.p', 'rb'))
# MTPVal = pickle.load(open(data_dir+'Validated_MTP_dict.p', 'rb'))
# main_pep = pickle.load(open(data_dir+'main_peptide_quant_dict.p', 'rb'))

#substr_dict = pickle.load(open(data_dir+'genome_substr_dict.p', 'rb'))
# val_mtp_dict = pickle.load(open(data_dir+'Ion_validated_MTP_dict.p','rb'))
# val_ev_dict = pickle.load(open(data_dir+'Validation_search_evidence_dict.p', 'rb'))

#print(MTPVal)
#print(qMTP)
#print(main_pep)
#print(dp_ev_dict)
#print(dp_dict)
#print(mtp_dict)
#print(ptm_dict)
print(hc_mtp_dict)
#print(val_mtp_dict)
#print(val_ev_dict)"""

#main_pep = pickle.load(open("/Users/alexa/OneDrive/Documents/2025_Maropakis_AAS/pipeline_output/Skin/AA_subs_pipeline/main_peptide_quant_dict.p", 'rb'))
#print(main_pep)

# val_mtp_dict = pickle.load(open(muscle_aa_subs_dir+'Ion_validated_MTP_dict.p','rb'))
# print(val_mtp_dict)

## 1. SAAP Detection

### Dataset Metrics

In [ ]:
# retain only SAAP observations with 100% data completeness across all columns
SAAP_quant_complete_df=SAAP_quant_df.dropna()
print(f'{len(SAAP_quant_complete_df):,} fully complete SAAP observations retained')
print(f'{100*(len(SAAP_quant_complete_df)/len(SAAP_quant_df)):.2f}% dataset completeness')
SAAP_quant_df = SAAP_quant_complete_df # replace read-in SAAP_quant_df with filtered version with complete data for downstream analyses

# if wanting SAAP_quant_df with missing values, skip this step and use original SAAP_quant_df read in from file above
# SAAP_quant_df = pd.read_excel(DEPDIR+'SAAP_precursor_reporter_quant_data.xlsx', index_col=0)
# #SAAP_precursor_reporter_quant_data.xlsx contains both precursor and reporter ion data for validated SAAPs
# print(SAAP_quant_df)
# SAAP_quant_df_list = [SAAP_quant_df.loc[SAAP_quant_df['Dataset']==ds] for ds in datasets]

In [ ]:
# N peptide IDs through filtering steps
# get number of samples in each dataset
n_samples_data = []
total = 0
for ds in datasets:
    sample_map = sample_map_list[datasets.index(ds)]
    samples = sample_map['sample_name'].values

    n_samples = len(samples)
    total += n_samples
    n_samples_data.append([ds, n_samples])
    print(ds, n_samples)
print(total)
n_samples_df = pd.DataFrame(n_samples_data, columns=['Dataset', 'N samples'])
n_samples_df.to_excel(OUTDIR+'N_samples_in_datasets.xlsx')

In [ ]:
def get_filter_df(data_dir, samples):
    """ 
    function that reads in output from decode pipeline to get number of peptides in each category
    Input: directory with output files for dataset, dataset samples/TMT set names
    output: dataframe with the number of peptides in each category
    """
    dp_ev_dict = pickle.load(open(data_dir+'DP_search_evidence_dict.p', 'rb'))
    dp_dict = pickle.load(open(data_dir+'DP_dict.p', 'rb'))
    mtp_dict = pickle.load(open(data_dir+'MTP_dict.p', 'rb'))
    ptm_dict = pickle.load(open(data_dir+'PTM_dict.p', 'rb'))
    hc_mtp_dict = pickle.load(open(data_dir+'qMTP_dict.p', 'rb')) # dictionary of putative substitutions that pass FDR correction
    substr_dict = pickle.load(open(data_dir+'genome_substr_dict.p', 'rb'))
    val_mtp_dict = pickle.load(open(data_dir+'Ion_validated_MTP_dict.p','rb'))
    
    rows = []
    for s in samples:
        seqs = [[x for y in list(mtp_dict[s]['mistranslated sequence'].values()) for x in y] for s in samples]
        seqs = [x for y in seqs for x in y]
        if 'all_6frame_seqs' in substr_dict.keys():
            homolog_seqs = substr_dict['all_6frame_seqs']
        else:
            homolog_seqs = substr_dict['all_frame6_seqs']
        seqs_hom = [x for x in seqs if x in homolog_seqs]
        
        n_main = len(dp_ev_dict[s]['Raw file'])
        n_dp = len(dp_dict[s]['Raw file'])
        n_ptm = len(ptm_dict[s]['Raw file'])
        n_aas = len(mtp_dict[s]['Raw file'])
        n_nohom = len([i for i,x in mtp_dict[s]['mistranslated sequence'].items() if all(y not in seqs_hom for y in x)])
        n_hc = len([i for i,x in hc_mtp_dict[s]['mistranslated sequence'].items() if all(y not in seqs_hom for y in x)])
        n_val = len(val_mtp_dict[s]['Raw file'])

        rows.append([s, n_main, n_dp, n_ptm, n_aas, n_nohom, n_hc, n_val])
    df = pd.DataFrame(rows, columns=['TMT set', 'Main peptides', 'DP', 'PTM', 'AAS', 'Non-homologous', 'High-confidence', 'Validated'])

    return(df)

In [ ]:
# create a dictionary containing the dataframes generated with the above function for each dataset
filter_dict = {}
for ds in datasets:
    print(ds)
    data_dir = data_dir_list[datasets.index(ds)]
    samples = samples_list[datasets.index(ds)]
    filter_dict[ds] = get_filter_df(data_dir, samples)

for ds in datasets:
    filter_dict[ds]['Dataset'] = [ds]*len(filter_dict[ds])
pickle.dump(filter_dict, open(OUTDIR+'Modified_peptide_filter_dict_DP2valSAAP.p', 'wb'))

#filter_dict = pickle.load(open(OUTDIR+'Modified_peptide_filter_dict_DP2valSAAP.p', 'rb'))
all_df = pd.concat([filter_dict[ds] for ds in datasets])

sum_dict = {}
for col in all_df.columns:
    if col not in ['TMT set', 'Dataset', 'Tissue']:
        sum_dict[col] = np.nansum(all_df[col].values)
        
print(sum_dict)

In [ ]:
# filtering modified peptides to candidate SAAP
barplot_rows = []
for i, row in all_df.iterrows():
    barplot_rows.append([row['Dataset'], row['Main peptides'], 'Main peptides', row['TMT set']])
    barplot_rows.append([row['Dataset'], row['DP'], 'Dependent peptides', row['TMT set']])
    barplot_rows.append([row['Dataset'], row['PTM'], 'Peptides with PTM', row['TMT set']])
    barplot_rows.append([row['Dataset'], row['High-confidence'], 'Candidate SAAP', row['TMT set']])

barplot_df = pd.DataFrame(barplot_rows, columns=['Dataset', 'N peptide IDs', 'Peptide type', 'TMT set'])

plt.figure(figsize=(6.5, 4))
sns.set(style='whitegrid', context='notebook', font_scale=1.1)
ax = sns.barplot(
    data=barplot_df, 
    x='Dataset', 
    y='N peptide IDs', 
    hue='Peptide type', 
    dodge=False, 
    edgecolor='black', 
    linewidth=0.5, 
    alpha=0.9
)

ax.set_yscale('log')
ax.set_ylim([100, 1e6])
ax.set_ylabel('# peptides$/$sample', fontsize=13)
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=40, labelsize=11)
ax.tick_params(axis='y', labelsize=11)

ax.grid(True, axis='y', linestyle='solid', linewidth=0.5, color='gray')
ax.grid(False, axis='x')

handles, labels = ax.get_legend_handles_labels()
new_labels = ['Main peptides', 'Dependent peptides', 'Peptides with PTM', 'Candidate SAAP']
ax.legend(
    handles=handles,
    labels=new_labels,
    title='',
    bbox_to_anchor=(0.5, 1.12),
    loc='center',
    ncol=2,
    fontsize=10,
    handletextpad=0.5,
    columnspacing=1,
    frameon=False
)

plt.tight_layout()
plt.savefig(PLOT_DIR + 'Dataset_filtering_barplot.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# N samples per dataset and age and plot 
plot_rows = []
plot_cols = ['Dataset', 'N samples', 'Sample type']
for ds in datasets:
    sample_map = sample_map_list[datasets.index(ds)]
    n_t6_samples = len(sample_map.loc[sample_map['Group']=='t06mo'])
    n_t15_samples = len(sample_map.loc[sample_map['Group']=='t15mo'])
    n_t24_samples = len(sample_map.loc[sample_map['Group']=='t24mo'])
    n_t30_samples = len(sample_map.loc[sample_map['Group']=='t30mo'])
    plot_rows.append([ds, n_t6_samples, '6 months'])
    plot_rows.append([ds, n_t15_samples, '15 months'])
    plot_rows.append([ds, n_t24_samples, '24 months'])
    plot_rows.append([ds, n_t30_samples, '30 months'])

plot_df = pd.DataFrame(plot_rows, columns=plot_cols)
sns.set_style('whitegrid')
fig,ax = plt.subplots(figsize=(5,3))
sns.barplot(data=plot_df, x='Dataset', y='N samples', hue='Sample type', palette=sns.color_palette(palette='Greys', n_colors=4))
ax.tick_params('both', labelsize=12)
plt.xlabel('')
plt.ylabel('# samples', fontsize=14)
plt.ylim([0,5])
plt.bar_label(plt.gca().containers[2])
plt.legend(ncol=1, bbox_to_anchor=(1.04,1.15), frameon=False, fontsize=13, handletextpad=0.1)
plt.savefig(PLOT_DIR+'AgedMouse_N_samples_per_dataset.pdf', bbox_inches='tight')



In [ ]:
# N SAAPs per dataset and age and plot
plot_rows=[]
plot_cols=['Dataset','N SAAPs','Sample type']

for ds in datasets:
    ds_df=SAAP_quant_df[SAAP_quant_df['Dataset']==ds]

    n_t6=ds_df.loc[ds_df['Sample Type']=='t06mo','MTP_seq'].nunique()
    n_t15=ds_df.loc[ds_df['Sample Type']=='t15mo','MTP_seq'].nunique()
    n_t24=ds_df.loc[ds_df['Sample Type']=='t24mo','MTP_seq'].nunique()
    n_t30=ds_df.loc[ds_df['Sample Type']=='t30mo','MTP_seq'].nunique()

    plot_rows.append([ds,n_t6,'6 months'])
    plot_rows.append([ds,n_t15,'15 months'])
    plot_rows.append([ds,n_t24,'24 months'])
    plot_rows.append([ds,n_t30,'30 months'])

plot_df=pd.DataFrame(plot_rows,columns=plot_cols)
sns.set_style('whitegrid')
fig,ax=plt.subplots(figsize=(4.5,3))
sns.barplot(data=plot_df,x='Dataset',y='N SAAPs',hue='Sample type',palette=sns.color_palette(palette='Greys',n_colors=4))
ax.tick_params('both',labelsize=12)
plt.xlabel('')
plt.ylabel('# unique SAAPs',fontsize=14)
plt.bar_label(plt.gca().containers[2])

plt.legend(ncol=1,bbox_to_anchor=(1.04,1.15),frameon=False,fontsize=13,handletextpad=0.1)
plt.savefig(PLOT_DIR+'AgedMouse_N_SAAPs_per_dataset.pdf',bbox_inches='tight')

In [ ]:
# N peptides detected per dataset and age from MaxQuant evidence.txt
plot_rows=[]
plot_cols=['Dataset','N peptides','Sample type']
for i,ds in enumerate(datasets):
    evidence_df=pd.read_csv(mq_dir_list[i]+'combined/txt/evidence.txt', sep='\t', low_memory=False)
    evidence_df=evidence_df.loc[evidence_df['PEP']<=0.01,: ]
    n_t6=evidence_df.loc[evidence_df['Experiment'].str.contains('t06mo',na=False),'Sequence'].nunique()
    n_t15=evidence_df.loc[evidence_df['Experiment'].str.contains('t15mo',na=False),'Sequence'].nunique()
    n_t24=evidence_df.loc[evidence_df['Experiment'].str.contains('t24mo',na=False),'Sequence'].nunique()
    n_t30=evidence_df.loc[evidence_df['Experiment'].str.contains('t30mo',na=False),'Sequence'].nunique()
    plot_rows.append([ds,n_t6,'6 months'])
    plot_rows.append([ds,n_t15,'15 months'])
    plot_rows.append([ds,n_t24,'24 months'])
    plot_rows.append([ds,n_t30,'30 months'])

plot_df=pd.DataFrame(plot_rows,columns=plot_cols)
sns.set_style('whitegrid')
fig,ax=plt.subplots(figsize=(4.5,3))

sns.barplot(
    data=plot_df,
    x='Dataset',
    y='N peptides',
    hue='Sample type',
    palette=sns.color_palette(palette='Greys',n_colors=4)
)
ax.tick_params('both',labelsize=12)
plt.xlabel('')
plt.ylabel('# unique peptides',fontsize=14)
plt.bar_label(plt.gca().containers[2])
plt.legend(
    ncol=1,
    bbox_to_anchor=(1.04,1.15),
    frameon=False,
    fontsize=13,
    handletextpad=0.1
)

plt.savefig(PLOT_DIR+'AgedMouse_N_peptides_per_dataset.pdf',bbox_inches='tight')

In [ ]:
# weighted SAAP data completeness per tissue
# if using SAAP_quant_df with 100% data completeness, should read 100% completeness for all tissues
# if using SAAP_quant_df with all observations, may read varying completeness across tissues
comp_cols=[
'Positional_probability',
'SAAP_PEP',
'BP_PEP',
'MTP_PrecInt',
'BP_PrecInt',
'Prec_RAAS',
'MTP_ReportInt',
'BP_ReportInt',
'Reporter_RAAS',
'BP_ReportInt_Norm',
'MTP_ReportInt_Norm'
]

comp_df=SAAP_quant_df.copy()
plot_rows=[]
for ds in sorted(comp_df['Dataset'].dropna().unique()):
    ds_df=comp_df[comp_df['Dataset']==ds]
    total_values=len(ds_df)*len(comp_cols)
    nonmissing_values=sum(
        ds_df[c].notna().sum()
        for c in comp_cols
    )
    completeness=100*(nonmissing_values/total_values)
    plot_rows.append([ds,completeness])

plot_df=pd.DataFrame(plot_rows,columns=['Dataset','Completeness'])
fig,ax=plt.subplots(figsize=(5,4))
sns.barplot(data=plot_df,x='Dataset',y='Completeness',color='#9E9E9E')
ax.tick_params('both',labelsize=12)
plt.xlabel('')
plt.ylabel('Weighted data completeness (%)',fontsize=14)
plt.ylim(0,100)
plt.title('SAAP dataset completeness across tissues',fontsize=14,pad=15)
plt.bar_label(ax.containers[0], fmt='%.1f%%', fontsize=11)

plt.savefig(PLOT_DIR+'SAAP_weighted_data_completeness_by_tissue.pdf', bbox_inches='tight')

### Post-translational modifications

In [ ]:
# plotting PTMs found
def get_ptm_df(ptm_dict, ds):
    """
    function to get a dataframe of PTMS x datasets from the datasets' PTM_dict.p
    """
    master_ptm_list = []
    for s,v in ptm_dict.items():
        s_ptm_dict = v['PTM']
        for ptm_list in s_ptm_dict.values():
            new_ptm_list = [x for x in ptm_list if x not in master_ptm_list]
            new_ptm_list = list(set(new_ptm_list))
            master_ptm_list = master_ptm_list + new_ptm_list

    heatmap_df = pd.DataFrame(index=master_ptm_list, columns=list(range(1,24)))
    for s, v in ptm_dict.items():
        if ds !='Healthy':
            s_int = int(s[1:])
        else:
            s_int=s
        ptm_list = list(v['PTM'].values())
        ptm_list = [x for y in ptm_list for x in y]
        ptm_count = Counter(ptm_list)
        for ptm, count in ptm_count.items():
            heatmap_df.loc[ptm, s_int] = count
            #heatmap_df.loc[ptm, s_int] = np.log10(count)
    heatmap_df.fillna(0, inplace=True)
    return(heatmap_df)

# get PTM dataframe for each dataset
for i in range(len(datasets)):
    print(datasets[i])
    data_dir =  data_dir_list[i]
    ptm_dict = pickle.load(open(data_dir+'PTM_dict.p', 'rb'))
    heatmap_df = get_ptm_df(ptm_dict, ds)
    heatmap_df.to_excel(DEPDIR+'PTM_heatmap_df.xlsx')
    
# create a dictionary with the PTM dataframes for each dataset
ptm_heatmap_dict = {}
for i,ds in enumerate(datasets):
    data_dir = data_dir_list[i]
    print("creating dictionary with PTM dataframes for each dataset")
    ptm_df = pd.read_excel(data_dir + 'PTM_heatmap_df.xlsx', index_col=0)
    ptm_df['avg'] = [np.mean(row.values) for i,row in ptm_df.iterrows()]
    ptm_df.sort_values('avg', ascending=False, inplace=True) # sort by frequency to extract top PTMs for plot
    
    ptm_df.index = [x[0].upper()+x[1:] for x in ptm_df.index]
    ptm_heatmap_dict[ds] = ptm_df
    
pickle.dump(ptm_heatmap_dict, open(DEPDIR+'PTM_heatmap_dict.p', 'wb'))
#ptm_heatmap_dict = pickle.load(open(nofilter_outdir+'PTM_heatmap_dict.p', 'rb'))


# extract the top PTMs by frequency into a dataframe for plotting
print("Extracting top PTMs by frequency into a plotting dataframe")
top20 = []
for ds, ds_df in ptm_heatmap_dict.items():
    if len(top20)==0:
        top20 = ds_df.index.values[0:40]
    else:
        top20new = ds_df.index.values[0:40]
        top20 = [x for x in top20 if x in top20new]
plot_df = pd.DataFrame(index=top20, columns=datasets)
for ptm in top20:
    for ds in datasets:
        heatmap_df = ptm_heatmap_dict[ds]
        plot_df.loc[ptm,ds] = heatmap_df.loc[ptm,'avg']
plot_df = plot_df.astype(float)

# need number of peptides IDd to normalize N each PTM per 1000 peptides
print("finding number of peptide IDs")
ds_metrics= pd.read_excel(DEPDIR+'dataset_metrics.xlsx') # dataframe created externally with number of peptides identified in the main seach of each dataset
print(ds_metrics.columns.tolist())  # Print column names exactly as they are

n_peptides_list = []
for ds in datasets:
    n_peptides = ds_metrics.loc[ds_metrics['Dataset']==ds, 'Peptides (evidence)'].values[0]
    n_peptides_list.append(n_peptides)

# normalize PTM plot df by per thousand peptides
print("normalizing PTM plot df by per thousand peptides")
scaled_plot_df = deepcopy(plot_df)
for i,ds in enumerate(datasets):
    scaled_plot_df[ds] = [x/(n_peptides_list[i]/1000) for x in scaled_plot_df[ds]]
scaled_plot_df.to_excel(DEPDIR+'PTM_heatmap_data.xlsx')

print("finished!")

# plot ptm heatmap
cg = sns.clustermap(
    data=scaled_plot_df,
    yticklabels=True,
    norm=LogNorm(),
    cbar_kws={
        'orientation': 'horizontal',
        'ticks': [0.01, 0.1, 1, 10]
    },
    figsize=(5.5, 5.5),
    cmap=sns.color_palette('plasma', as_cmap=True),
    method='ward',
    col_cluster=False,
    row_cluster=True,
    dendrogram_ratio=(0.05, 0.1),
    cbar_pos=(0.55, 0.05, 0.35, 0.04)
)

cg.ax_cbar.set_title('PTMs per thousand peptides', fontsize=12, pad=6)
cg.ax_cbar.tick_params(labelsize=10)
cg.ax_cbar.minorticks_off() 

plt.setp(cg.ax_heatmap.xaxis.get_majorticklabels(), rotation=90, fontsize=12)
plt.setp(cg.ax_heatmap.yaxis.get_majorticklabels(), fontsize=11)
cg.ax_row_dendrogram.set_visible(False)

plt.savefig(PLOT_DIR + 'PTM_heatmap_top20.pdf', bbox_inches='tight')

### Number of unique validated SAAPs

In [ ]:
# functions to get number of unique candidate substituted peptides and number of unique validated substituted peptides for each sample in each dataset
def get_n_unq_seqs(data_dir, s):
    """ input: dataset directory, sample
        output: number of unique candidate substituted peptides
    """
    hc_mtp_dict = pickle.load(open(data_dir+'qMTP_dict.p', 'rb'))
    s_seqs = [x for y in hc_mtp_dict[s]['mistranslated sequence'].values() for x in y]
    s_seqs = list(set(s_seqs))
    n_seqs = len(s_seqs)
    return(n_seqs)

def get_n_unq_val(data_dir, s):
    """ input: dataset directory, sample
        output: number of unique validated substituted peptides
    """
    val_mtp_dict = pickle.load(open(data_dir+'Ion_validated_MTP_dict.p', 'rb'))
    s_seqs = list(val_mtp_dict[s]['mistranslated sequence'].values())
    s_seqs = list(set(s_seqs))
    n_seqs = len(s_seqs)
    return(n_seqs)

# get dataframe with number of unique substituted peptides, candidate and validated
rows = []
cols = ['Dataset', 'N SAAP sequences', 'SAAP sequence type', 'TMT set']

for i,ds in enumerate(datasets):
    print(ds)
    samples = samples_list[i]
    data_dir = data_dir_list[i]
    for s in samples:
        n_seqs = get_n_unq_seqs(data_dir, s)
        n_val_seqs = get_n_unq_val(data_dir, s)
        rows.append([ds, n_seqs, 'Candidate', s])
        rows.append([ds, n_val_seqs, 'Validated', s])
plt_df = pd.DataFrame(rows, columns=cols)

# get dataframe with percentage of peptides validated and percentage of peptides with no genome homology (from ds_filter_dict.p generated above)
pcnt_rows = []
for i,ds in enumerate(datasets):
    print(ds)
    data_dir = data_dir_list[i]
    samples = samples_list[i]
    ds_plt_df = plt_df.loc[plt_df['Dataset']==ds,:]
    ds_filter_dict = filter_dict[ds]
    for j,s in enumerate(samples):
        s_df = ds_plt_df.loc[ds_plt_df['TMT set']==s,:]
        n_cand = s_df.loc[s_df['SAAP sequence type']=='Candidate', 'N SAAP sequences'].values[0]
        n_val = s_df.loc[s_df['SAAP sequence type']=='Validated', 'N SAAP sequences'].values[0]
        val_pcnt = 100*(n_val/n_cand)
        n_aas = ds_filter_dict['AAS'][j]
        pcnt_rows.append([ds,s, val_pcnt, 'Validated SAAP'])
pcnt_df = pd.DataFrame(pcnt_rows, columns=['Dataset', 'TMT set', '% peptides', '% type'])
nonrectum_row = [i for i,row in pcnt_df.iterrows() if (row['TMT set']!='rectum') and (row['TMT set']!='bonemarrow')] # too little data, removed these tissues from analysis
pcnt_df = pcnt_df.loc[nonrectum_row]
pcnt_df.to_excel(DEPDIR+'percent_validated_genomehomol_SAAP.xlsx')
# pcnt_df = pd.read_excel(DEPDIR+'percent_validated_genomehomol_SAAP.xlsx', index_col=0)

# plot percentage of candidate SAAPs validated across samples
sns.set(style="whitegrid")  
fig, ax = plt.subplots(figsize=(5, 4))  
c_palette = {'Validated SAAP': '#9E9E9E'}
avg_pcnt_df = pcnt_df.groupby(['Dataset', 'TMT set', '% type'], as_index=False).agg({'% peptides': 'mean'})
sns.barplot(data=avg_pcnt_df, x='Dataset', y='% peptides', hue='% type', ax=ax, dodge=True,
            palette = c_palette, edgecolor='white', linewidth=1.5)
ax.get_legend().remove()
plt.ylim(0, 65) 
plt.yticks(fontsize=12)  
plt.xticks(rotation=45, fontsize=12)  
plt.xlabel('', fontsize=14)  
plt.title('SAAP Validation Rates Across Mouse Tissues')
plt.ylabel('% Candidate SAAPs Validated', fontsize=14)  
plt.tight_layout()

plt.savefig(PLOT_DIR + 'percent_validated_allDS_boxplot.pdf', bbox_inches='tight')
plt.show()

### PEP + positional probability distributions

In [ ]:
# positional probability distribution
fig,ax=plt.subplots(figsize=(5,4))
loc_df=SAAP_quant_df['Positional_probability']
loc_df=(loc_df).replace([np.inf,-np.inf],np.nan).dropna()
sns.histplot(x=loc_df,bins=40,color='#0D0887')
ax.tick_params('both',labelsize=13)
plt.xlabel('Positional probability',fontsize=14)
plt.ylabel('Count',fontsize=14)
plt.title('All AAS Across Aged Mouse Tissues',fontsize=14)
plt.savefig(PLOT_DIR+'loc_prob_all_AAS.pdf',bbox_inches='tight')

# positional probability correlation across tissues
loc_df=SAAP_quant_df.copy()
loc_df=loc_df.replace([np.inf,-np.inf],np.nan).dropna(subset=['Positional_probability'])
tissue_corr=loc_df.groupby('Dataset')['Positional_probability'].median().reset_index()
# tissue_corr=loc_df.groupby('Dataset')['Positional_probability'].mean().reset_index()
tissue_corr['Dataset_num']=range(len(tissue_corr))
r,p=sp.stats.pearsonr(tissue_corr['Dataset_num'],tissue_corr['Positional_probability'])

fig,ax=plt.subplots(figsize=(5,4))
sns.regplot(data=tissue_corr,x='Dataset_num',y='Positional_probability',color='#0D0887')
ax.tick_params('both',labelsize=13)
ax.set_xticks(tissue_corr['Dataset_num'])
ax.set_xticklabels(tissue_corr['Dataset'],rotation=45)
plt.xlabel('Tissue',fontsize=14)
plt.ylabel('Median positional probability',fontsize=14)
# plt.ylabel('Mean positional probability',fontsize=14)
plt.title(f'SAAP Localization Confidence Across Tissues\nr={r:.2f},p={p:.3g}',fontsize=14)
plt.savefig(PLOT_DIR+'loc_prob_tissue_correlation.pdf',bbox_inches='tight')

In [ ]:
# SAAP PEP distribution
fig,ax=plt.subplots(figsize=(5,4))
pep_df=SAAP_quant_df['SAAP_PEP']
pep_df=(pep_df).replace([np.inf,-np.inf],np.nan).dropna()
sns.histplot(x=pep_df,bins=40,color='#6A00A8')
ax.tick_params('both',labelsize=13)
plt.xlabel('PEP',fontsize=14)
plt.ylabel('Count',fontsize=14)
plt.title('All AAS Across Aged Mouse Tissues',fontsize=14)
plt.savefig(PLOT_DIR+'SAAP_PEP_distribution.pdf',bbox_inches='tight')

# SAAP PEP correlation across tissues
pep_df=SAAP_quant_df.copy()
pep_df=pep_df.replace([np.inf,-np.inf],np.nan).dropna(subset=['SAAP_PEP'])
tissue_corr=pep_df.groupby('Dataset')['SAAP_PEP'].median().reset_index()
tissue_corr['Dataset_num']=range(len(tissue_corr))
r,p=sp.stats.pearsonr(tissue_corr['Dataset_num'],tissue_corr['SAAP_PEP'])

fig,ax=plt.subplots(figsize=(5,4))
sns.regplot(data=tissue_corr,x='Dataset_num',y='SAAP_PEP',color='#6A00A8')
ax.tick_params('both',labelsize=13)
ax.set_xticks(tissue_corr['Dataset_num'])
ax.set_xticklabels(tissue_corr['Dataset'],rotation=45)
plt.xlabel('Tissue',fontsize=14)
plt.ylabel('Median PEP',fontsize=14)
plt.title(f'SAAP Identification Confidence Across Tissues\nr={r:.2f},p={p:.3g}',fontsize=14)
plt.savefig(PLOT_DIR+'SAAP_PEP_tissue_correlation.pdf',bbox_inches='tight')

In [ ]:
# SAAP vs BP PEP distribution by tissue
datasets=sorted(SAAP_quant_df['Dataset'].dropna().unique())
ncols=4
nrows=int(np.ceil(len(datasets)/ncols))
fig,axes=plt.subplots(figsize=(5*ncols,4*nrows),nrows=nrows,ncols=ncols)
axes=np.array(axes).flatten()
for ax,ds in zip(axes,datasets):
    df=SAAP_quant_df[SAAP_quant_df['Dataset']==ds].drop_duplicates(subset=['MTP_seq','tmt_set'])
    saap_pep=-np.log10(df['SAAP_PEP']).replace([np.inf,-np.inf],np.nan).dropna()
    bp_pep=-np.log10(df['BP_PEP']).replace([np.inf,-np.inf],np.nan).dropna()
    sns.kdeplot(x=bp_pep,label='BP',linewidth=2,linestyle='--',color='#9E9E9E',ax=ax)
    sns.kdeplot(x=saap_pep,label='SAAP',linewidth=2,color='#B12A90',ax=ax)
    ax.tick_params('both',labelsize=13)
    plt.sca(ax)
    plt.xlabel('-log$_{10}$ PEP',fontsize=14)
    plt.ylabel('Density',fontsize=14)
    plt.title(ds,fontsize=14)
    plt.legend(fontsize=10,frameon=False)
for ax in axes[len(datasets):]:
    ax.axis('off')
plt.tight_layout()
plt.savefig(PLOT_DIR+'SAAP_BP_PEP_kde_by_tissue.pdf',bbox_inches='tight')

In [ ]:
# number of fragment ions supporting substitution site
# plot n fragments barplots
plot_df = pd.read_excel(DEPDIR+'by_fragments_per_saap_4barplot_allDS.xlsx', index_col=0)
sns.set_style('whitegrid')
fig, ax = plt.subplots(figsize=(4, 4))
sns.barplot(data=plot_df, x='Bin', y='Count', hue='Used', dodge=False, palette=['#9E9E9E', '#B12A90'])
ax.tick_params('both', labelsize=14)

plt.ylabel('# peptides', fontsize=15)
handles, labels = ax.get_legend_handles_labels()
plt.yscale("log")
plt.legend(handles=handles, loc='upper right', labels=['Excluded', 'Included'], fontsize=13)
plt.xlabel('# peptide fragments supporting\nsubstitution site', fontsize=15)
plt.savefig(PLOT_DIR + 'fragment_ion_evidence_barplot_allDS.pdf', bbox_inches='tight')

# fragment ion support correlation across tissues
frag_df=pos_prob_df.copy()
frag_df['N_fragments']=frag_df[['bp_b_left_frag_mass','bp_b_right_frag_mass','bp_y_left_frag_mass','bp_y_right_frag_mass']].notna().sum(axis=1)
frag_df=frag_df.drop_duplicates(subset=['SAAP','TMT set'])
tissue_corr=frag_df.groupby('Dataset')['N_fragments'].mean().reset_index()
# tissue_corr=frag_df.groupby('Dataset')['N_fragments'].median().reset_index()
tissue_corr['Dataset_num']=range(len(tissue_corr))
r,p=sp.stats.pearsonr(tissue_corr['Dataset_num'],tissue_corr['N_fragments'])
fig,ax=plt.subplots(figsize=(5,4))
sns.regplot(data=tissue_corr,x='Dataset_num',y='N_fragments',color='#B12A90')
ax.tick_params('both',labelsize=13)
ax.set_xticks(tissue_corr['Dataset_num'])
ax.set_xticklabels(tissue_corr['Dataset'],rotation=45)
plt.xlabel('Tissue',fontsize=14)
plt.ylabel('Mean supporting fragments',fontsize=14)
# plt.ylabel('Median supporting fragments',fontsize=14)
plt.title(f'Fragment Ion Support Across Tissues\nr={r:.2f},p={p:.3g}',fontsize=14)
plt.savefig(PLOT_DIR+'fragment_ion_evidence_tissue_correlation.pdf',bbox_inches='tight')

### PEP post-FDR correction

In [ ]:
# cumulative density of q-values and PEPs across all datasets and samples
# plot the p-values determined by MaxQuant and the q-values determined from only the substituted peptides
plot_rows = []
plot_cols = ['Dataset', 'TMT/Tissue', 'q-value', 'PEP']
for ds in datasets:
    data_dir = data_dir_list[datasets.index(ds)]
    samples = samples_list[datasets.index(ds)]
    qmtp_dict = pickle.load(open(data_dir+'qMTP_dict.p', 'rb'))
    for s in samples:
        s_dict = qmtp_dict[s]
        for k,v in s_dict['Posterior subs probability'].items():
            plot_rows.append([ds, s, 1-v[0], s_dict['q-value'][k][0]])
plot_df = pd.DataFrame(plot_rows, columns=plot_cols)

plot_df['-log q'] = plot_df.apply(lambda x: -np.log10(x['q-value']),axis=1)
plot_df['-log PEP'] = plot_df.apply(lambda x: -np.log10(x['PEP']),axis=1)
fig,ax = plt.subplots(figsize=(2.5,2.5))
sns.ecdfplot(plot_df, x='-log q', label='log$_{10}$(q)', color='#9E9E9E')
sns.ecdfplot(plot_df, x='-log PEP', label='log$_{10}$(PEP)', color='#B12A90')
plt.ylabel('Cumulative density', fontsize=14)
plt.xlabel('-log$_{10}$(FDR)', fontsize=14)
ax.tick_params('both', labelsize=13)
plt.plot((2,2), plt.ylim(), '--r')
plt.legend(loc='lower right', fontsize=12, handletextpad=0.1)
plt.savefig(PLOT_DIR+'qvalue_CumDensity_plot.pdf', bbox_inches='tight')

In [ ]:
# cumulative density of q-values and PEPs across all datasets and samples stratified by RAAS
# generate a dictionary with PEP values and N fragments for all SAAP in all datasets 
paas_pep_dict = {}
for ds in datasets:
    print(ds)
    data_dir = data_dir_list[datasets.index(ds)]
    samples = samples_list[datasets.index(ds)]
    mtp_dict = pickle.load(open(data_dir + 'Ion_validated_MTP_dict.p', 'rb'))
    mtp_quant_dict = pickle.load(open(data_dir + 'MTP_quant_dict.p', 'rb'))

    plot_rows = []
    plot_cols = ['Dataset', 'Sample', 'PAAS', 'PAAS_val_idx', 'DP PEP', 'PAAS PEP', 'PAAS q', 'PAAS Precursor Intensity','BP Precursor Intensity', 'Precursor RAAS', 'N fragments']
    for s in samples:
        s_dict = mtp_dict[s]
        for k, paas in s_dict['mistranslated sequence'].items():
            paas_pep = s_dict['Posterior subs probability'][k]
            paas_q = s_dict['q-value'][k]
            dp_pep = s_dict['DP PEP'][k]
            n_frags = s_dict['fragment_evidence'][k]
            q_dict = [i for i,v in mtp_quant_dict.items() if v['MTP_seq']==paas]
            if len(q_dict)>0:
                q_dict = mtp_quant_dict[q_dict[0]]
                paas_prec = q_dict['MTP_PrecInt'][s]
                bp_prec = q_dict['BP_PrecInt'][s]
                raas = q_dict['Prec_ratio'][s]
                if (~np.isnan(raas)) and (~np.isinf(raas)):
                    plot_rows.append([ds, s, paas, k, dp_pep, paas_pep, paas_q, paas_prec, bp_prec, raas, n_frags])
    plot_df = pd.DataFrame(plot_rows, columns=plot_cols)   
    paas_pep_dict[ds] = plot_df

pickle.dump(paas_pep_dict, open(OUTDIR+'SAAP_PEP_dfs.p', 'wb'))

# create data frame for plot 
all_ds_plot_df = pd.concat([v for k,v in paas_pep_dict.items()])
all_ds_plot_df['RAAS group'] = ['$\geq$0' if raas>=0 else '<0' for i,raas in list(enumerate(all_ds_plot_df['Precursor RAAS'].values))]
all_ds_plot_df = all_ds_plot_df.loc[all_ds_plot_df['PAAS q']<=0.01]

# plot histogram of confidence values stratified by RAAS
fig,ax = plt.subplots(figsize=(5,4))
all_ds_plot_df['-log q'] = [-np.log10(x) for x in all_ds_plot_df['PAAS q']]
high_raas_df = all_ds_plot_df.loc[all_ds_plot_df['Precursor RAAS']>=0]
low_raas_df = all_ds_plot_df.loc[all_ds_plot_df['Precursor RAAS']<0]
sns.kdeplot(data = high_raas_df.loc[~np.isinf(high_raas_df['-log q'])], x='-log q', fill=False, color='#9E9E9E', label='log$_{10}$(RAAS)$\geq$0')
sns.kdeplot(data = low_raas_df.loc[~np.isinf(low_raas_df['-log q'])], x='-log q', fill=False, label='log$_{10}$(RAAS) < 0', color='#B12A90')
ax.tick_params('both', labelsize=13)
plt.ylabel('Density', fontsize=14)
plt.xlabel('-log q-value', fontsize=14)
plt.legend(fontsize=12)
plt.savefig(PLOT_DIR+'RAASgroup_logq_hist.pdf', bbox_inches='tight')

# plot histogram of N fragments on substitution site stratified by RAAS
fig,ax = plt.subplots(figsize=(5,4))
sns.kdeplot(data = high_raas_df.loc[~np.isinf(high_raas_df['-log q'])], x='N fragments', fill=False, color='#9E9E9E', label='log$_{10}$(RAAS)$\geq$0')
sns.kdeplot(data = low_raas_df.loc[~np.isinf(low_raas_df['-log q'])], x='N fragments', fill=False, label='log$_{10}$(RAAS) < 0', color='#B12A90')
ax.tick_params('both', labelsize=13)
plt.ylabel('Density', fontsize=14)
plt.xlabel('# fragments supporting substitution', fontsize=14)
plt.yticks([0,0.03,0.06,0.09,0.12])
plt.legend(loc='upper right', fontsize=12)
plt.savefig(PLOT_DIR+'RAASgroup_Nfrags_hist.pdf', bbox_inches='tight')

## 2. Analysis of Substitution Types


### Shared Setup for Substitution Analyses


In [ ]:
from itertools import combinations
from scipy.stats import fisher_exact, kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

raas_col = 'Reporter_RAAS'
age_order = ['t06mo', 't15mo', 't24mo', 't30mo']
replicate_keep = [1, 2, 3, 4]

# Amino acids are ordered by broad biochemical class for matrix-style plots.
aa_order = [
    'G', 'A', 'V', 'L', 'I', 'M', 'P',
    'F', 'Y', 'W',
    'S', 'T', 'C', 'N', 'Q',
    'D', 'E',
    'K', 'R', 'H'
]


def add_age_tissue_replicate_columns(df):
    """Add common analysis columns used throughout substitution-type analyses."""
    out = df.copy()
    out['Age'] = out['Sample Type']
    out['Tissue'] = out['Dataset']
    out['Replicate'] = out['Sample'].str.extract(r'_rep(\d+)_', expand=False)
    out['Replicate'] = pd.to_numeric(out['Replicate'], errors='coerce')
    return out


def filter_standard_aging_replicates(df):
    """Keep the age groups and technical replicates used in the aging analyses."""
    out = add_age_tissue_replicate_columns(df)
    out = out[
        out['Age'].isin(age_order)
        &
        out['Replicate'].isin(replicate_keep)
    ].copy()

    if out.empty:
        raise ValueError(
            "No rows remain after filtering ages/replicates. "
            "Check Sample and Sample Type columns."
        )

    return out


def top_substitution_order(df, n=15):
    """Return the top substitutions in descending observed frequency."""
    return df['aa_sub'].value_counts().head(n).index


def add_fdr_column(df, p_col='pval', out_col='FDR'):
    """Apply BH-FDR correction while preserving NaN p-values."""
    df[out_col] = np.nan
    valid_pvals = df[p_col].notna()

    if valid_pvals.any():
        df.loc[valid_pvals, out_col] = multipletests(
            df.loc[valid_pvals, p_col],
            method='fdr_bh'
        )[1]

    return df


def add_significance_labels(df, fdr_col='FDR'):
    """Convert FDR thresholds to plotting/reporting labels."""
    df['Significance'] = 'ns'
    df.loc[df[fdr_col] < 0.05, 'Significance'] = '*'
    df.loc[df[fdr_col] < 0.01, 'Significance'] = '**'
    df.loc[df[fdr_col] < 0.001, 'Significance'] = '***'
    return df


def safe_filename_label(value):
    """Make dataset labels safe to use inside output filenames."""
    return str(value).replace(' ', '_').replace('/', '_')


def plot_raas_box_strip(
    plot_df,
    x_col,
    hue_col,
    y_col,
    x_order,
    hue_order,
    palette_dict,
    figsize,
    xlabel,
    ylabel,
    title,
    output_name,
    tick_rotation=0,
    strip_jitter=0.10,
    strip_size=4.2,
    strip_linewidth=0.55,
    legend_title='AAS',
    legend_fontsize=12,
    legend_title_fontsize=13,
    x_fontsize=17,
    y_fontsize=17,
    title_fontsize=20,
    tick_fontsize=15,
    y_tick_fontsize=14,
    show_and_close=False
):
    """Draw the repeated RAAS boxplot plus overlaid stripplot figure."""
    fig, ax = plt.subplots(figsize=figsize)

    sns.boxplot(
        data=plot_df,
        x=x_col,
        y=y_col,
        hue=hue_col,
        order=x_order,
        hue_order=hue_order,
        palette=palette_dict,
        showfliers=False,
        width=0.8,
        linewidth=1.1,
        ax=ax
    )

    sns.stripplot(
        data=plot_df,
        x=x_col,
        y=y_col,
        hue=hue_col,
        order=x_order,
        hue_order=hue_order,
        palette=palette_dict,
        dodge=True,
        jitter=strip_jitter,
        size=strip_size,
        alpha=0.75,
        edgecolor='white',
        linewidth=strip_linewidth,
        ax=ax,
        legend=False,
        zorder=10 if strip_size <= 4.2 else 12
    )

    handles, labels = ax.get_legend_handles_labels()

    ax.legend(
        handles[:len(hue_order)],
        labels[:len(hue_order)],
        title=legend_title,
        bbox_to_anchor=(1.02, 1),
        loc='upper left',
        frameon=True,
        fontsize=legend_fontsize,
        title_fontsize=legend_title_fontsize
    )

    ax.tick_params(axis='x', rotation=tick_rotation, labelsize=tick_fontsize)
    ax.tick_params(axis='y', labelsize=y_tick_fontsize)
    ax.set_xlabel(xlabel, fontsize=x_fontsize)
    ax.set_ylabel(ylabel, fontsize=y_fontsize)
    ax.set_title(title, fontsize=title_fontsize)
    ax.grid(axis='y', color='lightgray', linewidth=0.6, alpha=0.7)

    plt.savefig(PLOT_DIR + output_name, bbox_inches='tight')

    if show_and_close:
        plt.show(fig)
        plt.close(fig)

    return fig, ax


def compute_raas_global_and_pairwise(
    plot_df,
    group_col,
    group_order,
    sub_order,
    group_label,
    context_cols=None,
    context_values=None
):
    """Run Kruskal tests across groups and pairwise Mann-Whitney U tests for each substitution."""
    context_cols = context_cols or []
    context_values = context_values or []

    global_results = []

    for sub in sub_order:
        sub_data = plot_df[plot_df['aa_sub'] == sub]
        groups = [
            sub_data.loc[sub_data[group_col] == group, raas_col].dropna()
            for group in group_order
        ]
        valid_groups = [g for g in groups if len(g) > 0]

        if len(valid_groups) >= 2:
            stat, pval = kruskal(*valid_groups)
        else:
            stat, pval = np.nan, np.nan

        global_results.append(
            context_values + [sub, stat, pval, sum(len(g) for g in valid_groups)]
        )

    global_df = pd.DataFrame(
        global_results,
        columns=context_cols + [
            'aa_sub',
            'Kruskal_statistic',
            'pval',
            'n_observations'
        ]
    )
    global_df = add_fdr_column(global_df)

    pairwise_results = []

    for sub in sub_order:
        sub_data = plot_df[plot_df['aa_sub'] == sub]

        for group1, group2 in combinations(group_order, 2):
            x = sub_data.loc[sub_data[group_col] == group1, raas_col].dropna()
            y = sub_data.loc[sub_data[group_col] == group2, raas_col].dropna()

            if len(x) > 0 and len(y) > 0:
                stat, pval = mannwhitneyu(x, y, alternative='two-sided')
            else:
                stat, pval = np.nan, np.nan

            pairwise_results.append(
                context_values + [
                    sub,
                    group1,
                    group2,
                    len(x),
                    len(y),
                    x.mean(),
                    y.mean(),
                    x.median(),
                    y.median(),
                    x.mean() - y.mean(),
                    x.median() - y.median(),
                    stat,
                    pval
                ]
            )

    pairwise_df = pd.DataFrame(
        pairwise_results,
        columns=context_cols + [
            'aa_sub',
            f'{group_label}_1',
            f'{group_label}_2',
            f'{group_label}_1_n',
            f'{group_label}_2_n',
            f'{group_label}_1_mean_RAAS',
            f'{group_label}_2_mean_RAAS',
            f'{group_label}_1_median_RAAS',
            f'{group_label}_2_median_RAAS',
            'Mean_RAAS_difference',
            'Median_RAAS_difference',
            'MannWhitneyU_statistic',
            'pval'
        ]
    )

    pairwise_df = add_significance_labels(add_fdr_column(pairwise_df))
    pairwise_df['Comparison'] = (
        pairwise_df[f'{group_label}_1']
        + ' vs '
        + pairwise_df[f'{group_label}_2']
    )
    pairwise_df = pairwise_df.sort_values(
        ['FDR', 'pval'],
        ascending=[True, True],
        na_position='last'
    )

    return global_df, pairwise_df


### Most Frequent Substitution Types


In [ ]:
# Count all observed SAAP events and plot the 15 most frequent substitutions.
sub_df = SAAP_quant_df.copy()
top_subs = top_substitution_order(sub_df, n=15)
plot_df = sub_df[sub_df['aa_sub'].isin(top_subs)]
sub_order = plot_df['aa_sub'].value_counts().index

fig, ax = plt.subplots(figsize=(5, 4))

sns.countplot(
    data=plot_df,
    y='aa_sub',
    order=sub_order,
    color='#FCA636',
    ax=ax
)

ax.tick_params('both', labelsize=12)
plt.xlabel('# SAAP Observations', fontsize=14)
plt.ylabel('AA Substitution', fontsize=14)
plt.title('Top 15 Most Frequent AAS', fontsize=14)
plt.savefig(PLOT_DIR + 'top15_SAAP_substitutions.pdf', bbox_inches='tight')


### Top Substitution Types Across Tissues


In [ ]:
# Test whether each top substitution is differentially represented between tissues.
sub_df = SAAP_quant_df.copy()
top_subs = top_substitution_order(sub_df, n=15)
plot_df = sub_df[sub_df['aa_sub'].isin(top_subs)].copy()

sub_order = plot_df['aa_sub'].value_counts().loc[top_subs].index
tissue_order = sorted(plot_df['Dataset'].unique())

# Count tissue/substitution combinations, explicitly retaining zero-count pairs.
full_index = pd.MultiIndex.from_product(
    [tissue_order, sub_order],
    names=['Dataset', 'aa_sub']
)

count_df = (
    plot_df.groupby(['Dataset', 'aa_sub'])
    .size()
    .reindex(full_index, fill_value=0)
    .reset_index(name='Count')
)

# Pairwise Fisher exact tests compare each substitution against all other substitutions.
tissue_totals = sub_df['Dataset'].value_counts()
sub_counts = sub_df.groupby(['Dataset', 'aa_sub']).size()
results = []

for sub in sub_order:
    for tissue1, tissue2 in combinations(tissue_order, 2):
        a = sub_counts.get((tissue1, sub), 0)
        c = sub_counts.get((tissue2, sub), 0)
        b = tissue_totals[tissue1] - a
        d = tissue_totals[tissue2] - c

        odds, p = fisher_exact([[a, b], [c, d]], alternative='two-sided')

        results.append({
            'aa_sub': sub,
            'Tissue_1': tissue1,
            'Tissue_2': tissue2,
            'Tissue_1_targetSub_count': a,
            'Tissue_1_allOtherSubs_count': b,
            'Tissue_2_targetSub_count': c,
            'Tissue_2_allOtherSubs_count': d,
            'Odds_ratio_[a/b]/[c/d]': odds,
            'pval': p,
        })

stats_df = pd.DataFrame(results)
stats_df['FDR'] = multipletests(stats_df['pval'], method='fdr_bh')[1]
stats_df['Significance'] = pd.cut(
    stats_df['FDR'],
    bins=[-float('inf'), 0.001, 0.01, 0.05, float('inf')],
    labels=['***', '**', '*', 'ns'],
    right=False,
)

# Add per-tissue fractions so effect sizes are interpretable alongside count tables.
for tissue in ['Tissue_1', 'Tissue_2']:
    target = f'{tissue}_targetSub_count'
    other = f'{tissue}_allOtherSubs_count'
    stats_df[f'{tissue}_sub_fraction'] = stats_df[target] / (
        stats_df[target] + stats_df[other]
    )

stats_df['Comparison'] = stats_df['Tissue_1'] + ' vs ' + stats_df['Tissue_2']
stats_df = stats_df[
    [
        'aa_sub',
        'Comparison',
        'Tissue_1',
        'Tissue_1_targetSub_count',
        'Tissue_1_allOtherSubs_count',
        'Tissue_1_sub_fraction',
        'Tissue_2',
        'Tissue_2_targetSub_count',
        'Tissue_2_allOtherSubs_count',
        'Tissue_2_sub_fraction',
        'Odds_ratio_[a/b]/[c/d]',
        'pval',
        'FDR',
        'Significance',
    ]
]

fig, ax = plt.subplots(figsize=(18, 8))

sns.barplot(
    data=count_df,
    x='Dataset',
    y='Count',
    hue='aa_sub',
    hue_order=sub_order,
    palette=sns.color_palette('husl', len(sub_order)),
    order=tissue_order,
    ax=ax,
)

ax.tick_params(axis='x', rotation=45, labelsize=14)
ax.tick_params(axis='y', labelsize=13)
ax.set_xlabel('Tissue', fontsize=16)
ax.set_ylabel('# SAAP Observations', fontsize=16)
ax.set_title('Top 15 AAS Across Tissues', fontsize=18)
ax.legend(
    title='AAS',
    bbox_to_anchor=(1.02, 1),
    loc='upper left',
    frameon=True,
    fontsize=12,
    title_fontsize=13,
)

plt.savefig(PLOT_DIR + 'top15_substitutions_across_tissues.pdf', bbox_inches='tight')

stats_df = stats_df.sort_values(['FDR', 'pval'])
stats_df.to_csv(
    OUTDIR + 'top15_substitution_pairwise_statistics.tsv',
    sep='\t',
    index=False,
)


### Global Substitution Matrix


In [ ]:
# Bubble size shows observed count; color shows P(Y|X); outline marks global enrichment.
sub_df = SAAP_quant_df.copy()

sub_split = sub_df['aa_sub'].str.split(' to ', expand=True)
sub_df['From_AA'] = sub_split[0]
sub_df['To_AA'] = sub_split[1]

count_df = (
    sub_df.groupby(['From_AA', 'To_AA'])
    .size()
    .reset_index(name='Count')
)
count_df = count_df[count_df['From_AA'] != count_df['To_AA']]
count_df['Row_fraction'] = count_df.groupby('From_AA')['Count'].transform(lambda x: x / x.sum())
count_df['From_AA'] = pd.Categorical(count_df['From_AA'], categories=aa_order, ordered=True)
count_df['To_AA'] = pd.Categorical(count_df['To_AA'], categories=aa_order, ordered=True)
count_df = count_df.sort_values(['From_AA', 'To_AA'])

mat = (
    count_df.pivot_table(index='From_AA', columns='To_AA', values='Count', fill_value=0)
    .reindex(index=aa_order, columns=aa_order, fill_value=0)
)

for aa in aa_order:
    mat.loc[aa, aa] = 0

obs = mat.values.astype(float)
row_tot = obs.sum(axis=1, keepdims=True)
col_tot = obs.sum(axis=0, keepdims=True)
grand = obs.sum()
expected = (row_tot @ col_tot) / grand

# Fisher exact tests compare observed swap counts with independence-based expected counts.
results = []

for i, from_aa in enumerate(aa_order):
    for j, to_aa in enumerate(aa_order):
        if from_aa == to_aa:
            continue

        observed = obs[i, j]
        expected_count = expected[i, j]
        row_fraction = observed / row_tot[i, 0] if row_tot[i, 0] > 0 else np.nan
        a = observed
        b = grand - a
        c = expected_count
        d = grand - c

        odds_ratio, pval = fisher_exact([[a, b], [c, d]], alternative='greater')

        results.append([
            from_aa,
            to_aa,
            observed,
            expected_count,
            row_fraction,
            a,
            b,
            c,
            d,
            odds_ratio,
            pval
        ])

stats_df = pd.DataFrame(
    results,
    columns=[
        'From_AA',
        'To_AA',
        'Observed_count',
        'Expected_count',
        'P_Y_given_X',
        'Observed_thisSwap',
        'Observed_allOtherSwaps',
        'Expected_thisSwap',
        'Expected_allOtherSwaps',
        'Odds_ratio',
        'pval'
    ]
)

reject, padj, _, _ = multipletests(stats_df['pval'], method='fdr_bh')
stats_df['FDR'] = padj
stats_df['Sig'] = stats_df['FDR'] < 0.01

plot_df = count_df.merge(stats_df, on=['From_AA', 'To_AA'], how='left')
plot_df['From_AA'] = pd.Categorical(plot_df['From_AA'], categories=aa_order, ordered=True)
plot_df['To_AA'] = pd.Categorical(plot_df['To_AA'], categories=aa_order, ordered=True)

fig, ax = plt.subplots(figsize=(13.5, 10))

scatter = ax.scatter(
    x=plot_df['To_AA'].cat.codes,
    y=plot_df['From_AA'].cat.codes,
    s=np.log10(plot_df['Observed_count'] + 1) * 250,
    c=plot_df['P_Y_given_X'],
    cmap='plasma',
    edgecolors=np.where(plot_df['Sig'], 'deepskyblue', 'lightgray'),
    linewidths=np.where(plot_df['Sig'], 2.8, 0.6)
)

ax.set_xticks(range(len(aa_order)))
ax.set_yticks(range(len(aa_order)))
ax.set_xticklabels(aa_order, fontsize=13)
ax.set_yticklabels(aa_order, fontsize=13)
ax.grid(color='lightgray', linewidth=0.5)
ax.tick_params('both', length=0)
plt.xlabel('Substituted Amino Acid (Y)', fontsize=15)
plt.ylabel('Original Amino Acid (X)', fontsize=15)
plt.title('AAS Matrix', fontsize=16)
fig.subplots_adjust(right=0.74)

cbar = plt.colorbar(scatter, pad=0.02)
cbar.set_label(
    r'$P(Y\mid X)$' + '\n' + 'Substituted AA (Y) probability given original AA (X)',
    fontsize=14
)
cbar.ax.tick_params(labelsize=12)

# Build a count-size legend that adapts to the observed substitution range.
max_count = int(plot_df['Observed_count'].max())

if max_count <= 25:
    size_vals = [1, 5, 10, 25]
elif max_count <= 100:
    size_vals = [1, 10, 25, 50, 100]
elif max_count <= 500:
    size_vals = [1, 10, 50, 100, 250, 500]
elif max_count <= 1000:
    size_vals = [1, 10, 50, 100, 250, 500, 1000]
else:
    size_vals = [1, 10, 50, 100, 250, 500, 1000, int(round(max_count, -3))]

size_vals = [v for v in size_vals if v <= max_count]
size_handles = [
    plt.scatter(
        [],
        [],
        s=np.log10(v + 1) * 250,
        facecolors='lightgray',
        edgecolors='black',
        linewidths=1
    )
    for v in size_vals
]
size_labels = [str(v) for v in size_vals]
sig_handle = plt.scatter([], [], s=120, facecolors='lightgray', edgecolors='deepskyblue', linewidths=2.8)
nonsig_handle = plt.scatter([], [], s=120, facecolors='lightgray', edgecolors='lightgray', linewidths=0.6)

legend = ax.legend(
    handles=size_handles + [sig_handle, nonsig_handle],
    labels=size_labels + ['FDR < 0.01', 'Not significant'],
    title='Size = Observed Sub Count\nOutline = Global enrichment\n(Fisher exact, BH-FDR corrected)',
    scatterpoints=1,
    frameon=True,
    fontsize=11,
    title_fontsize=12,
    loc='upper left',
    bbox_to_anchor=(1.32, 1),
    labelspacing=1.2
)

plt.savefig(PLOT_DIR + 'SAAP_substitution_bubble_matrix.pdf', bbox_inches='tight')

stats_df = stats_df.sort_values(['FDR', 'Observed_count'], ascending=[True, False])
stats_df.to_csv(
    OUTDIR + 'SAAP_substitution_statistics.tsv',
    sep='\t',
    index=False
)


### RAAS Across Age for Globally Common Substitutions


In [ ]:
# Compare Reporter_RAAS across age groups for the globally most common substitutions.
sub_df = filter_standard_aging_replicates(SAAP_quant_df)
top_subs = top_substitution_order(sub_df, n=15)
plot_df = sub_df[sub_df['aa_sub'].isin(top_subs)].copy()

if plot_df.empty:
    raise ValueError("plot_df is empty after selecting top substitutions.")

sub_order = plot_df['aa_sub'].value_counts().loc[top_subs].index
raas_age_global_df, raas_age_pairwise_df = compute_raas_global_and_pairwise(
    plot_df=plot_df,
    group_col='Age',
    group_order=age_order,
    sub_order=sub_order,
    group_label='Age'
)

plot_df['Age'] = pd.Categorical(plot_df['Age'], categories=age_order, ordered=True)
plot_df['aa_sub'] = pd.Categorical(plot_df['aa_sub'], categories=sub_order, ordered=True)
palette_dict = dict(zip(sub_order, sns.color_palette('husl', len(sub_order))))

plot_raas_box_strip(
    plot_df=plot_df,
    x_col='Age',
    hue_col='aa_sub',
    y_col=raas_col,
    x_order=age_order,
    hue_order=sub_order,
    palette_dict=palette_dict,
    figsize=(20, 9),
    xlabel='Age',
    ylabel='RAAS',
    title='Top 15 AAS RAAS Across Ages',
    output_name='top15_substitution_ReporterRAAS_across_ages_pooled_replicates_with_points.pdf'
)

raas_age_global_df.to_csv(
    OUTDIR + 'top15_substitution_ReporterRAAS_age_global_statistics.tsv',
    sep='\t',
    index=False
)

raas_age_pairwise_df.to_csv(
    OUTDIR + 'top15_substitution_ReporterRAAS_age_pairwise_statistics.tsv',
    sep='\t',
    index=False
)


### RAAS Across Tissues for Globally Common Substitutions


In [ ]:
# Compare Reporter_RAAS across tissues for the globally most common substitutions.
sub_df = add_age_tissue_replicate_columns(SAAP_quant_df)
sub_df = sub_df[sub_df['Replicate'].isin(replicate_keep)].copy()

if sub_df.empty:
    raise ValueError(
        "No rows remain after filtering replicates. "
        "Check Sample column."
    )

top_subs = top_substitution_order(sub_df, n=15)
plot_df = sub_df[sub_df['aa_sub'].isin(top_subs)].copy()

if plot_df.empty:
    raise ValueError("plot_df is empty after selecting top substitutions.")

sub_order = plot_df['aa_sub'].value_counts().loc[top_subs].index
tissue_order = sorted(plot_df['Tissue'].dropna().unique())
raas_tissue_global_df, raas_tissue_pairwise_df = compute_raas_global_and_pairwise(
    plot_df=plot_df,
    group_col='Tissue',
    group_order=tissue_order,
    sub_order=sub_order,
    group_label='Tissue'
)

plot_df['Tissue'] = pd.Categorical(plot_df['Tissue'], categories=tissue_order, ordered=True)
plot_df['aa_sub'] = pd.Categorical(plot_df['aa_sub'], categories=sub_order, ordered=True)

# View 1: substitutions are colored within each tissue.
palette_dict = dict(zip(sub_order, sns.color_palette('husl', len(sub_order))))
plot_raas_box_strip(
    plot_df=plot_df,
    x_col='Tissue',
    hue_col='aa_sub',
    y_col=raas_col,
    x_order=tissue_order,
    hue_order=sub_order,
    palette_dict=palette_dict,
    figsize=(24, 9),
    xlabel='Tissue',
    ylabel='RAAS',
    title='Top 15 AAS RAAS Across Tissues',
    output_name='top15_substitution_ReporterRAAS_across_tissues_pooled_replicates_with_points.pdf',
    tick_rotation=45
)

# View 2: tissues are grouped within each substitution.
palette_dict = dict(zip(tissue_order, sns.color_palette('plasma', len(tissue_order))))
plot_raas_box_strip(
    plot_df=plot_df,
    x_col='aa_sub',
    hue_col='Tissue',
    y_col=raas_col,
    x_order=sub_order,
    hue_order=tissue_order,
    palette_dict=palette_dict,
    figsize=(30, 20),
    xlabel='AAS',
    ylabel='RAAS',
    title='RAAS Across Tissues Grouped by AAS',
    output_name='top15_substitution_ReporterRAAS_tissues_grouped_by_AAS_with_points.pdf',
    tick_rotation=45,
    strip_jitter=0.06,
    strip_size=5.5,
    strip_linewidth=0.65,
    legend_title='Tissue',
    legend_fontsize=30,
    legend_title_fontsize=35,
    x_fontsize=45,
    y_fontsize=45,
    title_fontsize=45,
    tick_fontsize=25,
    y_tick_fontsize=25
)

raas_tissue_global_df.to_csv(
    OUTDIR + 'top15_substitution_ReporterRAAS_tissue_global_statistics.tsv',
    sep='\t',
    index=False
)

raas_tissue_pairwise_df.to_csv(
    OUTDIR + 'top15_substitution_ReporterRAAS_tissue_pairwise_statistics.tsv',
    sep='\t',
    index=False
)


### Tissue Differences Within Each Age Group


In [ ]:
# Repeat the tissue comparison separately within each age group.
sub_df = filter_standard_aging_replicates(SAAP_quant_df)
top_subs = top_substitution_order(sub_df, n=15)
plot_base_df = sub_df[sub_df['aa_sub'].isin(top_subs)].copy()

if plot_base_df.empty:
    raise ValueError("plot_base_df is empty after selecting top substitutions.")

sub_order = plot_base_df['aa_sub'].value_counts().loc[top_subs].index
tissue_order = sorted(plot_base_df['Tissue'].dropna().unique())

global_results = []
pairwise_results = []

for age in age_order:
    age_df = plot_base_df[plot_base_df['Age'] == age]
    age_global_df, age_pairwise_df = compute_raas_global_and_pairwise(
        plot_df=age_df,
        group_col='Tissue',
        group_order=tissue_order,
        sub_order=sub_order,
        group_label='Tissue',
        context_cols=['Age'],
        context_values=[age]
    )
    global_results.append(age_global_df)
    pairwise_results.append(age_pairwise_df)

raas_age_tissue_global_df = pd.concat(global_results, ignore_index=True)
raas_age_tissue_pairwise_df = pd.concat(pairwise_results, ignore_index=True)

plot_base_df['aa_sub'] = pd.Categorical(plot_base_df['aa_sub'], categories=sub_order, ordered=True)
plot_base_df['Tissue'] = pd.Categorical(plot_base_df['Tissue'], categories=tissue_order, ordered=True)
plot_base_df['Age'] = pd.Categorical(plot_base_df['Age'], categories=age_order, ordered=True)
palette_dict = dict(zip(tissue_order, sns.color_palette('plasma', len(tissue_order))))

for age in age_order:
    plot_df = plot_base_df[plot_base_df['Age'] == age].copy()

    if plot_df.empty:
        continue

    plot_raas_box_strip(
        plot_df=plot_df,
        x_col='aa_sub',
        hue_col='Tissue',
        y_col=raas_col,
        x_order=sub_order,
        hue_order=tissue_order,
        palette_dict=palette_dict,
        figsize=(30, 20),
        xlabel='AAS',
        ylabel='RAAS',
        title='RAAS Across Tissues Grouped by AAS - ' + age,
        output_name='top15_substitution_ReporterRAAS_tissues_grouped_by_AAS_' + age + '_with_points.pdf',
        tick_rotation=45,
        strip_jitter=0.06,
        strip_size=5.5,
        strip_linewidth=0.65,
        legend_title='Tissue',
        legend_fontsize=30,
        legend_title_fontsize=35,
        x_fontsize=45,
        y_fontsize=45,
        title_fontsize=45,
        tick_fontsize=25,
        y_tick_fontsize=25,
        show_and_close=True
    )

raas_age_tissue_global_df.to_csv(
    OUTDIR + 'top15_substitution_ReporterRAAS_tissue_global_statistics_by_age.tsv',
    sep='\t',
    index=False
)

raas_age_tissue_pairwise_df.to_csv(
    OUTDIR + 'top15_substitution_ReporterRAAS_tissue_pairwise_statistics_by_age.tsv',
    sep='\t',
    index=False
)


### Age-by-Tissue RAAS Interaction Summary


In [ ]:
# Collapse to SAAP-level means, then summarize tissue trajectories across age for each substitution.
sub_df = filter_standard_aging_replicates(SAAP_quant_df)
top_subs = top_substitution_order(sub_df, n=15)
sub_df = sub_df[sub_df['aa_sub'].isin(top_subs)].copy()

if sub_df.empty:
    raise ValueError("No rows remain after selecting top substitutions.")

sub_order = sub_df['aa_sub'].value_counts().loc[top_subs].index
tissue_order = sorted(sub_df['Tissue'].dropna().unique())

saap_mean_df = (
    sub_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=[raas_col])
    .groupby(
        ['Tissue', 'Age', 'aa_sub', 'MTP_seq', 'BP_seq'],
        observed=False
    )[raas_col]
    .mean()
    .reset_index()
    .rename(columns={raas_col: 'Mean_Reporter_RAAS'})
)

saap_mean_df['Age'] = pd.Categorical(saap_mean_df['Age'], categories=age_order, ordered=True)
saap_mean_df['aa_sub'] = pd.Categorical(saap_mean_df['aa_sub'], categories=sub_order, ordered=True)
saap_mean_df['Tissue'] = pd.Categorical(saap_mean_df['Tissue'], categories=tissue_order, ordered=True)
palette_dict = dict(zip(tissue_order, sns.color_palette('plasma', len(tissue_order))))

# Line plot: median SAAP-level Reporter_RAAS trajectories by tissue, faceted by substitution.
g = sns.relplot(
    data=saap_mean_df,
    x='Age',
    y='Mean_Reporter_RAAS',
    hue='Tissue',
    col='aa_sub',
    col_wrap=5,
    kind='line',
    estimator='median',
    errorbar=('ci', 95),
    marker='o',
    markersize=7,
    linewidth=2.2,
    hue_order=tissue_order,
    col_order=sub_order,
    palette=palette_dict,
    height=4.2,
    aspect=1.2,
    facet_kws={'sharey': True}
)

for ax in g.axes.flat:
    ax.tick_params(axis='x', rotation=45, labelsize=11)
    ax.tick_params(axis='y', labelsize=11)
    ax.set_xlabel('Age', fontsize=12)
    ax.set_ylabel('Median SAAP-level Reporter RAAS', fontsize=12)
    ax.grid(axis='y', color='lightgray', linewidth=0.5, alpha=0.7)

g.fig.suptitle(
    'Age-Dependent Tissue Differences in Reporter RAAS by AAS',
    fontsize=20,
    y=1.02
)

g.fig.savefig(
    PLOT_DIR + 'top15_substitution_ReporterRAAS_age_tissue_interaction_by_AAS.pdf',
    bbox_inches='tight'
)

plt.show()
plt.close(g.fig)

# Heatmap: range of tissue median Reporter_RAAS values for each substitution and age.
summary_df = (
    saap_mean_df
    .groupby(['aa_sub', 'Age', 'Tissue'], observed=False)['Mean_Reporter_RAAS']
    .median()
    .reset_index()
)

spread_df = (
    summary_df
    .groupby(['aa_sub', 'Age'], observed=False)['Mean_Reporter_RAAS']
    .agg(lambda x: x.max() - x.min())
    .reset_index(name='Tissue_RAAS_range')
)

spread_mat = (
    spread_df
    .pivot(index='aa_sub', columns='Age', values='Tissue_RAAS_range')
    .reindex(index=sub_order, columns=age_order)
)

fig, ax = plt.subplots(figsize=(7, 9))

sns.heatmap(
    spread_mat,
    cmap='magma',
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'Tissue range in median Reporter RAAS'},
    ax=ax
)

ax.set_title('Magnitude of Tissue Differences Across Age', fontsize=16, pad=14)
ax.set_xlabel('Age', fontsize=13)
ax.set_ylabel('AAS', fontsize=13)
ax.tick_params(axis='x', rotation=0, labelsize=12)
ax.tick_params(axis='y', labelsize=11)

plt.savefig(
    PLOT_DIR + 'top15_substitution_ReporterRAAS_tissue_difference_range_heatmap_by_age.pdf',
    bbox_inches='tight'
)

plt.show()
plt.close(fig)

saap_mean_df.to_csv(
    OUTDIR + 'top15_substitution_SAAP_level_mean_ReporterRAAS_by_tissue_age.tsv',
    sep='\t',
    index=False
)

spread_df.to_csv(
    OUTDIR + 'top15_substitution_ReporterRAAS_tissue_difference_range_by_age.tsv',
    sep='\t',
    index=False
)


### Age Trends Within Each Tissue


In [ ]:
# For each tissue, compare Reporter_RAAS across age groups using that tissue's top substitutions.
sub_df = filter_standard_aging_replicates(SAAP_quant_df)
tissue_order = sorted(sub_df['Tissue'].dropna().unique())

all_global_results = []
all_pairwise_results = []

for tissue in tissue_order:
    tissue_df = sub_df[sub_df['Tissue'] == tissue].copy()

    if tissue_df.empty:
        continue

    top_subs = top_substitution_order(tissue_df, n=15)
    plot_df = tissue_df[tissue_df['aa_sub'].isin(top_subs)].copy()

    if plot_df.empty:
        continue

    sub_order = plot_df['aa_sub'].value_counts().loc[top_subs].index
    tissue_global_df, tissue_pairwise_df = compute_raas_global_and_pairwise(
        plot_df=plot_df,
        group_col='Age',
        group_order=age_order,
        sub_order=sub_order,
        group_label='Age',
        context_cols=['Tissue'],
        context_values=[tissue]
    )

    all_global_results.append(tissue_global_df)
    all_pairwise_results.append(tissue_pairwise_df)

    plot_df['Age'] = pd.Categorical(plot_df['Age'], categories=age_order, ordered=True)
    plot_df['aa_sub'] = pd.Categorical(plot_df['aa_sub'], categories=sub_order, ordered=True)
    palette_dict = dict(zip(sub_order, sns.color_palette('husl', len(sub_order))))

    plot_raas_box_strip(
        plot_df=plot_df,
        x_col='Age',
        hue_col='aa_sub',
        y_col=raas_col,
        x_order=age_order,
        hue_order=sub_order,
        palette_dict=palette_dict,
        figsize=(20, 9),
        xlabel='Age',
        ylabel='RAAS',
        title=tissue + ' Top 15 AAS RAAS Across Ages',
        output_name='top15_substitution_ReporterRAAS_across_ages_' + safe_filename_label(tissue) + '_with_points.pdf',
        show_and_close=True
    )

raas_tissue_age_global_df = pd.concat(all_global_results, ignore_index=True)
raas_tissue_age_pairwise_df = pd.concat(all_pairwise_results, ignore_index=True)

raas_tissue_age_global_df.to_csv(
    OUTDIR + 'top15_substitution_ReporterRAAS_age_global_statistics_by_tissue.tsv',
    sep='\t',
    index=False
)

raas_tissue_age_pairwise_df.to_csv(
    OUTDIR + 'top15_substitution_ReporterRAAS_age_pairwise_statistics_by_tissue.tsv',
    sep='\t',
    index=False
)


### RAAS Fold Change Relative to t06mo


In [ ]:
# Compute SAAP-level log2 fold change in mean RAAS relative to each SAAP's t06mo baseline.
ctrl_label = 't06mo'
plot_age_order = ['t15mo', 't24mo', 't30mo']
peptide_keys = ['Dataset', 'MTP_seq', 'BP_seq', 'aa_sub']

sub_df = filter_standard_aging_replicates(SAAP_quant_df)
sub_df['Group'] = sub_df['Age']

# Reporter_RAAS is already log10 RAAS.
sub_df['log10_RAAS'] = sub_df[raas_col]
sub_df = sub_df.replace([np.inf, -np.inf], np.nan).dropna(subset=['log10_RAAS'])

saap_age_mean_df = (
    sub_df
    .groupby(peptide_keys + ['Group'], observed=False)['log10_RAAS']
    .mean()
    .reset_index()
    .rename(columns={'log10_RAAS': 'mean_log10_RAAS'})
)

ctrl_df = (
    saap_age_mean_df[saap_age_mean_df['Group'] == ctrl_label]
    [peptide_keys + ['mean_log10_RAAS']]
    .rename(columns={'mean_log10_RAAS': 'mean_log10_RAAS_t06mo'})
)

norm_df = saap_age_mean_df.merge(ctrl_df, on=peptide_keys, how='left')
norm_df['log2FC_vs_t06mo'] = (
    norm_df['mean_log10_RAAS']
    -
    norm_df['mean_log10_RAAS_t06mo']
) * np.log2(10)

norm_df = norm_df.dropna(subset=['log2FC_vs_t06mo']).copy()

if norm_df.empty:
    raise ValueError(
        "No rows remain after t06mo normalization. "
        "Some SAAPs may be missing t06mo baseline values."
    )

norm_df['Tissue'] = norm_df['Dataset']
tissue_order = sorted(norm_df['Tissue'].dropna().unique())

for tissue in tissue_order:
    tissue_df = norm_df[norm_df['Tissue'] == tissue].copy()

    if tissue_df.empty:
        continue

    top_subs = (
        tissue_df[['MTP_seq', 'BP_seq', 'aa_sub']]
        .drop_duplicates()
        ['aa_sub']
        .value_counts()
        .head(15)
        .index
    )

    plot_df = tissue_df[
        tissue_df['aa_sub'].isin(top_subs)
        &
        tissue_df['Group'].isin(plot_age_order)
    ].copy()

    if plot_df.empty:
        continue

    sub_order = (
        tissue_df[tissue_df['aa_sub'].isin(top_subs)]
        ['aa_sub']
        .value_counts()
        .loc[top_subs]
        .index
    )

    plot_df['aa_sub'] = pd.Categorical(plot_df['aa_sub'], categories=sub_order, ordered=True)
    plot_df['Group'] = pd.Categorical(plot_df['Group'], categories=plot_age_order, ordered=True)
    palette_dict = dict(zip(plot_age_order, sns.color_palette('Set2', len(plot_age_order))))

    fig, ax = plt.subplots(figsize=(18, 8))

    sns.boxplot(
        data=plot_df,
        x='aa_sub',
        y='log2FC_vs_t06mo',
        hue='Group',
        order=sub_order,
        hue_order=plot_age_order,
        palette=palette_dict,
        showfliers=False,
        width=0.75,
        linewidth=1.3,
        ax=ax
    )

    sns.stripplot(
        data=plot_df,
        x='aa_sub',
        y='log2FC_vs_t06mo',
        hue='Group',
        order=sub_order,
        hue_order=plot_age_order,
        palette=palette_dict,
        dodge=True,
        jitter=0.08,
        size=4.6,
        alpha=0.75,
        edgecolor='white',
        linewidth=0.6,
        ax=ax,
        legend=False,
        zorder=10
    )

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(
        handles[:len(plot_age_order)],
        labels[:len(plot_age_order)],
        title='Age',
        bbox_to_anchor=(1.02, 1),
        loc='upper left',
        frameon=True,
        fontsize=13,
        title_fontsize=14
    )

    ax.axhline(0, color='black', linestyle='--', linewidth=1.2)
    ax.tick_params(axis='x', rotation=45, labelsize=14)
    ax.tick_params(axis='y', labelsize=14)
    ax.set_xlabel('AAS', fontsize=17, labelpad=10)
    ax.set_ylabel(r'log$_2$(mean RAAS at age / mean RAAS at t06mo)', fontsize=17, labelpad=10)
    ax.set_title(
        tissue + r': RAAS fold change per AAS relative to t06mo',
        fontsize=20,
        pad=16
    )
    ax.grid(axis='y', color='lightgray', linewidth=0.7, alpha=0.75)

    plt.savefig(
        PLOT_DIR + 'top15_substitution_log2_mean_RAAS_over_t06mo_' + safe_filename_label(tissue) + '_no_t06mo.pdf',
        bbox_inches='tight'
    )

    plt.show()
    plt.close(fig)

norm_df.to_csv(
    OUTDIR + 'SAAP_level_log2_mean_RAAS_over_t06mo.tsv',
    sep='\t',
    index=False
)


In [ ]:
import itertools

ctrl_label='t06mo'
plot_age_order=['t15mo','t24mo','t30mo']
peptide_keys=['Dataset','MTP_seq','BP_seq','aa_sub']

sub_df=filter_standard_aging_replicates(SAAP_quant_df)
sub_df['Group']=sub_df['Age']
sub_df['log10_RAAS']=sub_df[raas_col]
sub_df=sub_df.replace([np.inf,-np.inf],np.nan).dropna(subset=['log10_RAAS'])

saap_age_mean_df=sub_df.groupby(
    peptide_keys+['Group'],
    observed=False
)['log10_RAAS'].mean().reset_index().rename(
    columns={'log10_RAAS':'mean_log10_RAAS'}
)

ctrl_df=saap_age_mean_df[
    saap_age_mean_df['Group']==ctrl_label
][peptide_keys+['mean_log10_RAAS']].rename(
    columns={'mean_log10_RAAS':'mean_log10_RAAS_t06mo'}
)

norm_df=saap_age_mean_df.merge(
    ctrl_df,
    on=peptide_keys,
    how='left'
)

norm_df['log2FC_vs_t06mo']=(
    norm_df['mean_log10_RAAS']
    -
    norm_df['mean_log10_RAAS_t06mo']
)*np.log2(10)

norm_df=norm_df.dropna(
    subset=['log2FC_vs_t06mo']
).copy()

if norm_df.empty:
    raise ValueError(
        'No rows remain after t06mo normalization.'
    )

norm_df['Tissue']=norm_df['Dataset']
norm_df=norm_df[
    norm_df['Group'].isin(plot_age_order)
]

top_subs=norm_df[
    ['MTP_seq','BP_seq','aa_sub']
].drop_duplicates()['aa_sub'].value_counts().head(20).index

plot_df=norm_df[
    norm_df['aa_sub'].isin(top_subs)
].copy()

sub_order=plot_df[
    'aa_sub'
].value_counts().loc[top_subs].index

plot_df['Tissue_Age']=(
    plot_df['Tissue']
    +
    '_'
    +
    plot_df['Group']
)

heat_df=plot_df.groupby(
    ['aa_sub','Tissue_Age'],
    observed=False
)['log2FC_vs_t06mo'].mean().reset_index().pivot(
    index='aa_sub',
    columns='Tissue_Age',
    values='log2FC_vs_t06mo'
)

col_order=[]

for t in sorted(plot_df['Tissue'].unique()):
    for a in plot_age_order:
        x=t+'_'+a
        if x in heat_df.columns:
            col_order.append(x)

heat_df=heat_df.reindex(
    index=sub_order,
    columns=col_order
)

fig,ax=plt.subplots(
    figsize=(max(18,len(col_order)*0.6),11)
)

vmax=np.nanpercentile(
    np.abs(heat_df.values),
    95
)

sns.heatmap(
    heat_df,
    cmap='coolwarm',
    center=0,
    vmin=-vmax,
    vmax=vmax,
    linewidths=.35,
    linecolor='white',
    cbar_kws={
        'label':
        r'Mean log$_2$(age RAAS/t06mo RAAS)'
    },
    ax=ax
)

age_labels=[
    x.split('_')[1]
    .replace('t','')
    .replace('mo',' mo')
    for x in heat_df.columns
]

ax.set_xticklabels(
    age_labels,
    rotation=45,
    ha='right',
    fontsize=13,
    fontstyle='italic'
)

plt.yticks(
    rotation=35,
    fontsize=12,
    fontstyle='italic'
)

tissues=[
    x.split('_')[0]
    for x in heat_df.columns
]

start=0

for tissue,grp in itertools.groupby(tissues):

    n=len(list(grp))

    left=start-.5
    right=start+n-.5
    center=(left+right)/2

    ax.plot(
        [left,right],
        [-0.22,-0.22],
        transform=ax.get_xaxis_transform(),
        color='black',
        lw=2,
        clip_on=False
    )

    ax.plot(
        [left,left],
        [-0.22,-0.15],
        transform=ax.get_xaxis_transform(),
        color='black',
        lw=2,
        clip_on=False
    )

    ax.plot(
        [right,right],
        [-0.22,-0.15],
        transform=ax.get_xaxis_transform(),
        color='black',
        lw=2,
        clip_on=False
    )

    ax.text(
        center,
        -0.25,
        tissue,
        transform=ax.get_xaxis_transform(),
        ha='center',
        va='top',
        fontsize=20
    )

    start+=n

ax.set_xlabel('')
ax.set_ylabel(
    'AA Substitution Type',
    fontsize=23,
)

ax.set_title(
    'RAAS Fold Change Relative to t06mo Across Tissues, Ages, and Swap Type',
    fontsize=23,
    pad=13
)

plt.subplots_adjust(bottom=.34)

plt.savefig(
    PLOT_DIR+'Global_RAAS_log2FC_heatmap_all_tissues.pdf',
    bbox_inches='tight'
)

plt.show()
plt.close()

### Biochemical Classes of Substitution Types


In [ ]:
# # Annotate substitutions by amino-acid chemistry and summarize age/tissue enrichment patterns.
# aa_class = {
#     'A': 'hydrophobic', 'V': 'hydrophobic', 'L': 'hydrophobic',
#     'I': 'hydrophobic', 'M': 'hydrophobic', 'F': 'aromatic',
#     'W': 'aromatic', 'Y': 'aromatic', 'S': 'polar',
#     'T': 'polar', 'N': 'polar', 'Q': 'polar',
#     'C': 'polar', 'G': 'special', 'P': 'special',
#     'K': 'positive', 'R': 'positive', 'H': 'positive',
#     'D': 'negative', 'E': 'negative'
# }

# charge_map = {
#     'hydrophobic': 'neutral',
#     'aromatic': 'neutral',
#     'polar': 'neutral',
#     'special': 'neutral',
#     'positive': 'positive',
#     'negative': 'negative'
# }

# conservative_pairs = {
#     frozenset(['D', 'E']),
#     frozenset(['K', 'R']),
#     frozenset(['S', 'T']),
#     frozenset(['N', 'Q']),
#     frozenset(['F', 'Y']),
#     frozenset(['I', 'L']),
#     frozenset(['V', 'I']),
#     frozenset(['V', 'L'])
# }

# annot_df = SAAP_quant_df.copy()
# annot_df = annot_df.drop_duplicates(subset=['MTP_seq', 'Dataset', 'Sample Type'])
# annot_df[['Ref_AA', 'Alt_AA']] = annot_df['aa_sub'].str.split(' to ', expand=True)
# annot_df['Ref_class'] = annot_df['Ref_AA'].map(aa_class)
# annot_df['Alt_class'] = annot_df['Alt_AA'].map(aa_class)
# annot_df['Chemical_change'] = annot_df['Ref_class'] + ' → ' + annot_df['Alt_class']
# annot_df['Charge_change'] = annot_df['Ref_class'].map(charge_map) + ' → ' + annot_df['Alt_class'].map(charge_map)
# annot_df['Conservative'] = annot_df.apply(
#     lambda x: frozenset([x['Ref_AA'], x['Alt_AA']]) in conservative_pairs,
#     axis=1
# )

# top_changes = annot_df['Chemical_change'].value_counts().head(15).index
# annot_df = annot_df[annot_df['Chemical_change'].isin(top_changes)]

# # Age enrichment: compare each biochemical class frequency with the t06mo baseline.
# global_counts = annot_df.groupby(['Sample Type', 'Chemical_change']).size().reset_index(name='N')
# global_counts['Frequency'] = global_counts['N'] / global_counts.groupby('Sample Type')['N'].transform('sum')
# rows = []

# for change in top_changes:
#     t6_freq = global_counts.loc[
#         (global_counts['Sample Type'] == 't06mo')
#         &
#         (global_counts['Chemical_change'] == change),
#         'Frequency'
#     ]
#     if len(t6_freq) == 0:
#         continue

#     t6_freq = t6_freq.values[0]
#     total_n = annot_df.loc[annot_df['Chemical_change'] == change, 'MTP_seq'].nunique()

#     for age in ['t15mo', 't24mo', 't30mo']:
#         age_freq = global_counts.loc[
#             (global_counts['Sample Type'] == age)
#             &
#             (global_counts['Chemical_change'] == change),
#             'Frequency'
#         ]
#         if len(age_freq) == 0:
#             continue

#         age_freq = age_freq.values[0]
#         log2_fc = np.log2(age_freq / t6_freq)
#         rows.append([
#             change,
#             age.replace('t', '').replace('mo', ' months'),
#             log2_fc,
#             total_n
#         ])

# global_enrich_df = pd.DataFrame(
#     rows,
#     columns=['Chemical change', 'Age comparison', 'log2_FC', 'Total abundance']
# )

# fig, ax = plt.subplots(figsize=(6.5, 6))

# sns.scatterplot(
#     data=global_enrich_df,
#     x='Age comparison',
#     y='Chemical change',
#     size='Total abundance',
#     hue='log2_FC',
#     sizes=(60, 700),
#     palette='plasma',
#     edgecolor='black',
#     linewidth=0.5,
#     legend=False,
#     ax=ax
# )

# norm = plt.Normalize(global_enrich_df['log2_FC'].min(), global_enrich_df['log2_FC'].max())
# sm = plt.cm.ScalarMappable(cmap='plasma', norm=norm)
# sm.set_array([])
# cbar = plt.colorbar(sm, ax=ax)
# cbar.set_label('log$_2$(group swap frequency / t06mo swap frequency)', fontsize=13)
# cbar.ax.tick_params(labelsize=11)
# ax.tick_params('both', labelsize=12)
# plt.xlabel('')
# plt.ylabel('Substitution Class', fontsize=14)
# plt.title('Age-associated Biochemical Substitution Enrichment', fontsize=14, pad=15)
# plt.savefig(PLOT_DIR + 'Biochemical_SAAP_age_enrichment.pdf', bbox_inches='tight')

# # Tissue enrichment: center each biochemical class by its mean frequency across tissues.
# tissue_only_counts = annot_df.groupby(['Dataset', 'Chemical_change']).size().reset_index(name='N')
# tissue_only_counts['Frequency'] = tissue_only_counts['N'] / tissue_only_counts.groupby('Dataset')['N'].transform('sum')
# heatmap_df = tissue_only_counts.pivot(index='Chemical_change', columns='Dataset', values='Frequency')
# heatmap_df = heatmap_df.loc[top_changes]
# heatmap_df = heatmap_df.sub(heatmap_df.mean(axis=1), axis=0)

# fig, ax = plt.subplots(figsize=(6, 5))

# sns.heatmap(
#     heatmap_df,
#     cmap='plasma',
#     center=0,
#     linewidths=0.5,
#     cbar_kws={'label': 'Mean Substitution Frequency'},
#     ax=ax
# )

# ax.tick_params('both', labelsize=11)
# plt.xlabel('Tissue', fontsize=14)
# plt.ylabel('Substitution Class', fontsize=14)
# plt.title('Cross-tissue Biochemical Substitution Enrichment', fontsize=14, pad=15)
# plt.savefig(PLOT_DIR + 'Cross_tissue_biochemical_SAAP_enrichment.pdf', bbox_inches='tight')

# # Conservative and nonconservative substitutions across aging.
# cons_counts = annot_df.groupby(['Sample Type', 'Conservative']).size().reset_index(name='N')
# cons_counts['Frequency'] = cons_counts['N'] / cons_counts.groupby('Sample Type')['N'].transform('sum')

# fig, ax = plt.subplots(figsize=(4, 4))

# sns.barplot(
#     data=cons_counts,
#     x='Sample Type',
#     y='Frequency',
#     hue='Conservative',
#     palette=['#aaaaaa', '#08A385'],
#     ax=ax
# )

# ax.tick_params('both', labelsize=12)
# plt.xlabel('Age group', fontsize=14)
# plt.ylabel('Normalized frequency', fontsize=14)
# plt.title('Conservative vs nonconservative substitutions across aging', fontsize=14, pad=15)
# plt.savefig(PLOT_DIR + 'Conservative_SAAPs_across_aging.pdf', bbox_inches='tight')

# # Tissue enrichment for the top individual amino-acid swaps.
# swap_df = annot_df.copy()
# swap_df['AA_swap'] = swap_df['Ref_AA'] + ' → ' + swap_df['Alt_AA']
# top_swaps = swap_df['AA_swap'].value_counts().head(20).index
# swap_df = swap_df[swap_df['AA_swap'].isin(top_swaps)]
# swap_counts = swap_df.groupby(['Dataset', 'AA_swap']).size().reset_index(name='N')
# swap_counts['Frequency'] = swap_counts['N'] / swap_counts.groupby('Dataset')['N'].transform('sum')
# heatmap_df = swap_counts.pivot(index='AA_swap', columns='Dataset', values='Frequency')
# heatmap_df = heatmap_df.loc[top_swaps]
# heatmap_df = heatmap_df.sub(heatmap_df.mean(axis=1), axis=0)

# fig, ax = plt.subplots(figsize=(6, 7))

# sns.heatmap(
#     heatmap_df,
#     cmap='plasma',
#     center=0,
#     linewidths=0.5,
#     cbar_kws={'label': 'Mean Substitution Frequency'},
#     ax=ax
# )

# ax.tick_params('both', labelsize=11)
# plt.xlabel('Tissue', fontsize=14)
# plt.ylabel('Substitution', fontsize=14)
# plt.title('Cross-tissue SAAP Swap enrichment', fontsize=14, pad=15)
# plt.savefig(PLOT_DIR + 'Cross_tissue_AA_swap_enrichment.pdf', bbox_inches='tight')

# annot_df.to_excel(OUTDIR + 'SAAP_biochemical_annotation_database.xlsx')
# global_enrich_df.to_excel(OUTDIR + 'Global_biochemical_SAAP_age_enrichment.xlsx')


## 3. RAAS Computations

In [ ]:
# sample-level RAAS distribution - all datasets in one distribution
sns.set_style('whitegrid')
fig,ax = plt.subplots(figsize=(3,3))
ratios = []
for i,ds in enumerate(datasets):
    ratio_data = [x for x in SAAP_quant_df_list[i]['Reporter_RAAS'].values if ~np.isnan(x) and ~np.isinf(x)]
    ratios = ratios + ratio_data
sns.histplot(ratios, color='#aaaaaa', linewidth=0, alpha=1)
plt.plot((np.median(ratios), np.median(ratios)),(plt.ylim()), '--r', color="red", linewidth=1.7)
ax.annotate('Median =\n$-$'+str(np.abs(np.round(np.median(ratios), 2))), (0,700), fontsize=10)
ax.set_ylabel(r'# SAAP', fontsize=14)
ax.set_xlabel(r'log$_{10}$(RAAS)', fontsize=14)
ax.tick_params('both',labelsize=13)
ax.set_xticks([-6,-4,-2,0,2,4]);
plt.savefig(PLOT_DIR+'Sample_level_RAAS_allDS_onehistplot.pdf', bbox_inches='tight')

In [ ]:
fig,ax=plt.subplots(figsize=(4,4))
raas_df=SAAP_quant_df.copy()
raas_df['log10_Prec_RAAS']=raas_df['Prec_RAAS'].replace([np.inf,-np.inf],np.nan)
raas_df['log10_Reporter_RAAS']=raas_df['Reporter_RAAS'].replace([np.inf,-np.inf],np.nan)
raas_df=raas_df.dropna(subset=['log10_Prec_RAAS','log10_Reporter_RAAS'])

r,p=sp.stats.pearsonr(
    raas_df['log10_Prec_RAAS'],
    raas_df['log10_Reporter_RAAS']
)

sns.scatterplot(
    data=raas_df,
    x='log10_Prec_RAAS',
    y='log10_Reporter_RAAS',
    s=18,
    alpha=1,
    color='#aaaaaa',
    edgecolor='white',
    linewidth=0.35,
    ax=ax
)

sns.regplot(
    data=raas_df,
    x='log10_Prec_RAAS',
    y='log10_Reporter_RAAS',
    scatter=False,
    line_kws={
        'color':'red',
        'linestyle':'--',
        'linewidth':2.5,
        'zorder':10
    },
    ax=ax
)

ax.tick_params('both',labelsize=13)
plt.xlabel('log$_{10}$(MS1-level RAAS)',fontsize=14)
plt.ylabel('log$_{10}$(MS2-level RAAS)',fontsize=14)
plt.title(f'MS1- vs MS2-Level RAAS\nr={r:.2f},p={p:.3g}',fontsize=14)
plt.savefig(PLOT_DIR+'MS1_MS2_RAAS_correlation.pdf',bbox_inches='tight')

In [ ]:
#TMT-level RAAS distribution (precursor ion RAAS distribution), code adapted from Tsour et al 2024
# N = number of quantified SAAP precursor-ion ratios

fig,axes = plt.subplots(1,len(datasets),figsize=(len(datasets),3), sharey=True)

sns.set_style('whitegrid')
for ax in axes:
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.tick_params('x', bottom=False,labelbottom=False)
    if ax!=axes[0]:    
        ax.spines['left'].set_visible(False)
        ax.tick_params(left=False)

medians = [np.nanmedian(SAAP_quant_df_list[i]['Prec_RAAS'].values) for i in range(len(datasets)-1)]
med_sort_idx = list(np.argsort(medians))
med_sort_idx.append(max(med_sort_idx)+1)

for i,ds in enumerate(datasets):
    ax_idx = med_sort_idx.index(i)
    axes[ax_idx].set_xlabel(ds, fontsize=14)
    ratio_data = [x for x in SAAP_quant_df_list[i]['Prec_RAAS'].values if ~np.isnan(x) and ~np.isinf(x)]
    bihist(ratio_data, ratio_data, nbins=50,h=axes[ax_idx])
    if ax_idx==0:
        axes[ax_idx].annotate('N= ', (-0.2,1.01), xycoords='axes fraction', fontsize=13)
    axes[ax_idx].annotate('{:,}'.format((len(ratio_data))), (0.2,1.01), xycoords='axes fraction', fontsize=13)
    axes[ax_idx].plot(axes[ax_idx].get_xlim(), (np.median(ratio_data), np.median(ratio_data)), '--r', color="red", linewidth=1.7)

axes[0].set_ylabel(r'log$_{10}$(RAAS)', fontsize=15)
axes[0].tick_params('y',labelsize=13)

plt.savefig(PLOT_DIR+'TMT_level_RAAS_allDS_nogrid.pdf', bbox_inches='tight')

In [ ]:
# sample/reporter ion level RAAS distributions for each dataset, code adapted from Tsour et al 2024
# N = number of quantified SAAP reporter-ion ratios

sns.set_style('whitegrid')
fig,axes = plt.subplots(1,len(datasets),figsize=(len(datasets),3), sharey=True)
plt.subplots_adjust(wspace=0.05)

for ax in axes:
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.tick_params('x', bottom=False,labelbottom=False)
    if ax!=axes[0]:    
        ax.spines['left'].set_visible(False)
        ax.tick_params(left=False)

medians = [np.nanmedian(SAAP_quant_df_list[i]['Reporter_RAAS'].values) for i in range(len(datasets)-1)]
med_sort_idx = list(np.argsort(medians))
med_sort_idx.append(max(med_sort_idx)+1)

for i,ds in enumerate(datasets):
    ax_idx = med_sort_idx.index(i)
    axes[ax_idx].set_xlabel(ds, fontsize=12)
    ratio_data = [x for x in SAAP_quant_df_list[i]['Reporter_RAAS'].values if ~np.isnan(x) and ~np.isinf(x)]
    bihist(ratio_data, ratio_data, nbins=100,h=axes[ax_idx])
    if ax_idx == 0:
        axes[ax_idx].annotate('N= ', (-0.1,1.01), xycoords='axes fraction', fontsize=13)
        axes[ax_idx].annotate(str(len(ratio_data)), (0.32,1.01), xycoords='axes fraction', fontsize=13)  
    else:
        axes[ax_idx].annotate(str(len(ratio_data)), (0.25,1.01), xycoords='axes fraction', fontsize=13)
        
    axes[ax_idx].plot(axes[ax_idx].get_xlim(), (np.median(ratio_data), np.median(ratio_data)), '--r', color="red", linewidth=1.7)

axes[0].set_ylabel(r'log$_{10}$(RAAS)', fontsize=15)
axes[0].tick_params('y',labelsize=13)
axes[0].set_yticks([-6,-4,-2,0,2,4]);

plt.savefig(PLOT_DIR+'Sample_level_RAAS_allDS.pdf', bbox_inches='tight')

In [ ]:
# add number of datasets each SAAP-BP pair is found in to data from SAAP_quant_df
saap_bp = [row['MTP_seq']+':'+row['BP_seq'] for i,row in SAAP_quant_df.iterrows()]
saap_bp_counter = Counter(saap_bp)
saap_bp_count_df = pd.DataFrame.from_dict(saap_bp_counter, columns=['N datasets'], orient='index')

def get_ds(saap, bp, SAAP_quant_df):
    """ function to get the list of datasets a given SAAP-BP pair is found in"""
    saap_bp_df = SAAP_quant_df.loc[(SAAP_quant_df['MTP_seq']==saap) & (SAAP_quant_df['BP_seq']==bp)]
    ds_list = saap_bp_df['Dataset'].tolist()
    return(ds_list)

SAAP_quant_df['N datasets'] = SAAP_quant_df.apply(lambda x: len(get_ds(x['MTP_seq'], x['BP_seq'], SAAP_quant_df)), axis=1)
SAAP_quant_df['Datasets'] = SAAP_quant_df.apply(lambda x: str(get_ds(x['MTP_seq'], x['BP_seq'], SAAP_quant_df)), axis=1)

# get subsets of the dataframes for which the SAAP-BP pair is found in 6+ datasets
n6_saap_df = SAAP_quant_df.loc[SAAP_quant_df['N datasets']>=6]

#  generate dataframe for heatmap showing in what percentage of samples does each shared saap come up in each dataset
shared_saap_bp_n6 = [i for i,row in saap_bp_count_df.iterrows() if row['N datasets']>=6]
n_samples_heatmap_df = pd.DataFrame(index=datasets, columns=shared_saap_bp_n6)
pcnt_samples_heatmap_df = pd.DataFrame(index=datasets, columns=shared_saap_bp_n6)

# dictionary with total number of samples in each dataset for computing percentages
n_total_samples_dict = {ds:0 for ds in datasets}
for ds in datasets:
    n_total_samples = len(samples_list[datasets.index(ds)])
    n_total_samples_dict[ds] = n_total_samples
        
for saap_bp in shared_saap_bp_n6:
    saap = saap_bp.split(':')[0]
    bp = saap_bp.split(':')[1]
    for ds in datasets:
        ds_prec_df = n6_saap_df.loc[n6_saap_df['Dataset']==ds]
        saap_bp_df = ds_prec_df.loc[(ds_prec_df['MTP_seq']==saap) & (ds_prec_df['BP_seq']==bp)]
        n_samples = len(saap_bp_df)
        pcnt_samples = 100*n_samples/n_total_samples_dict[ds]
        n_samples_heatmap_df.loc[ds, saap_bp] = n_samples
        pcnt_samples_heatmap_df.loc[ds, saap_bp] = pcnt_samples

#plot the heatmap
pcnt_samples_heatmap_df = pcnt_samples_heatmap_df.astype(float)
c = sns.clustermap(pcnt_samples_heatmap_df, figsize=(5,5), method='ward', xticklabels=False, vmin=0, vmax=80)
c.ax_row_dendrogram.set_visible(False)
c.ax_heatmap.set_xlabel('SAAP in $\geq$6 datasets', fontsize=15)
c.ax_heatmap.set_yticklabels(c.ax_heatmap.get_ymajorticklabels(), fontsize = 14)
c.ax_heatmap.set_facecolor('#F3F3F3')
c.ax_heatmap.yaxis.set_ticks_position('left')
c.ax_cbar.set_ylabel(r'% samples with SAAP', labelpad=3)
c.ax_cbar.set_position([0.95,0.06,0.03,0.7])
c.ax_cbar.yaxis.label.set_size(15)
c.ax_cbar.tick_params(labelsize=13)
c.savefig(PLOT_DIR+'%_samples_all_ds_sharedSAAPn6_vmax80.pdf', bbox_inches='tight')

In [ ]:
#RAAS distributions for shared SAAP (found in 6+ datasets)
fig,axes = plt.subplots(1,len(datasets),figsize=(len(datasets),3), sharey=True)
plt.subplots_adjust(wspace=0.05)

sns.set_style('whitegrid')
for ax in axes:
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)
    ax.tick_params('x', bottom=False,labelbottom=False)
    if ax!=axes[0]:    
        ax.spines['left'].set_visible(False)
        ax.tick_params(left=False)
        
# compute medians to sort datasets by
medians = [np.nanmedian(n6_saap_df.loc[n6_saap_df['Dataset']==ds,'Prec_RAAS'].values) for ds in datasets[:-1]]
med_sort_idx = list(np.argsort(medians))
med_sort_idx.append(max(med_sort_idx)+1)

# plot violin plots
for i,ds in enumerate(datasets):
    ax_idx = med_sort_idx.index(i)
    axes[ax_idx].set_xlabel(ds, fontsize=14)
    ratio_data = [x for x in n6_saap_df.loc[n6_saap_df['Dataset']==ds,'Prec_RAAS'].values if ~np.isnan(x) and ~np.isinf(x)]
    bihist(ratio_data, ratio_data, nbins=10,h=axes[ax_idx])
    if ax_idx==0:
        axes[ax_idx].annotate('N= ', (-0.2,1.01), xycoords='axes fraction', fontsize=13)

    axes[ax_idx].annotate(str(len(ratio_data)), (0.21,1.01), xycoords='axes fraction', fontsize=13)
        
    axes[ax_idx].plot(axes[ax_idx].get_xlim(), (np.median(ratio_data), np.median(ratio_data)), '--', color='red',linewidth=1.7)

axes[0].set_ylabel(r'log$_{10}$ RAAS', fontsize=15)
axes[0].tick_params('y',labelsize=13)
axes[0].set_yticks([-6,-4,-2,0,2,4]);
plt.savefig(PLOT_DIR+'Sample_level_RAAS_allDS_SharedSAAPin6.pdf', bbox_inches='tight')